# Offline replay, three arms: 1 kHz, 5 kHz, 5 kHz + low-pass

This notebook replays one closed-loop PRIME session offline three times. The arms differ only in how the
post-stimulus window is processed. Everything else is the online code in `prime/` (`prime_core`), imported
read-only, on the same saved session.

| arm | post-stimulus window, in calibration and on every pulse | stands for |
|---|---|---|
| **A** | resampled to 1 kHz; the resampler's anti-alias filter is its only low-pass (−6 dB at 500 Hz) | the decider as it ran |
| **B** | kept at 5 kHz; nothing low-passes it | the 5 kHz patch alone |
| **C** | kept at 5 kHz; a frozen 500 Hz zero-phase low-pass right after the first artifact blank | the 5 kHz patch + the low-pass |

The pre-stimulus window (QC and model input) and the ICA window stay at 1 kHz in every arm.

**How the code changes without touching `prime/`.** The 5 kHz patch (`post_5khz.patch`, commit `ee41369` on the
`post-5khz` branch) and the low-pass patch (`post_lowpass.patch`) are applied to the source text of the four modules
they touch while Python imports them (section 1.2). Nothing is written to `prime/`, not even `__pycache__`. A patch
the prime code on this machine already contains is skipped.

The arm whose processing matches the session (arm A for every session recorded so far) has to reproduce the online
log; section 2.3 checks the calibration, 3.2 the single pulses. Whatever it does not reproduce is replay noise, the
floor for reading B and C. Each arm is calibrated on its own: bad channels, ICA blink gate, SOUND, SSP-SIR, MAD
statistics, dipole, normaliser and PRIME model.

| section | content |
|---|---|
| 0 | parameters |
| 1 | setup: paths, the online code with the patches in memory, the online session, completeness check |
| 2 | calibration replay per arm; the online calibration; does the control arm reproduce it |
| 3 | intervention replay per arm (continual adaptation) |
| 4 | calibration compared across arms |
| 5 | rejections by reason |
| 6 | TEP amplitudes across arms, processing time per pulse |
| 7 | prediction performance |
| 8 | out-of-sample test on the evaluation blocks |

**What the replay cannot change.** Every pulse happens where it happened online, on a trial the online model and
threshold chose. The replay tests what each arm computes, and what its re-calibrated model predicts and learns, on
those trials; not which trials it would have selected.

**Requirements.** CUDA for the PRIME model (`ALLOW_CPU` is a fallback, `RUN_MODEL = False` skips the model),
`torch`, `mne`, `mne-icalabel`, and the folder `offline_replay_three_arm/` next to this notebook with
`post_lowpass.py`, `post_5khz.patch` and `post_lowpass.patch`. Three calibrations and three intervention replays:
expect several minutes per session.

## 0 · Parameters

All paths follow from `SUBJECT_ID`. `RUN_ID = None` picks the run folder that holds `prime/trials_intervention.csv`.
Output folders are created when missing. `SESSION_OVERRIDES` holds decider settings that changed between sessions:
the value a session ran with, where it differs from the prime code on this machine. Section 2.3 says when a session
needs one.

In [ ]:
# ---------------------------------------------------------------- session
SUBJECT_ID = "pilot005"          # BIDS subject label, without "sub-"
SESSION_ID = "ses-prime"
RUN_ID     = None                # NeuroSimo run folder, e.g. "105"; None = the one folder with prime/trials_intervention.csv

# ---------------------------------------------------------------- arms
ARMS_TO_RUN          = ["A", "B", "C"]   # any subset: A = 1 kHz, B = 5 kHz, C = 5 kHz + low-pass
LOWPASS_HZ           = 500.0             # arm C: cutoff of the post-stimulus low-pass (Hz)
LOWPASS_HALFWIDTH_MS = 5.0               # arm C: reach on each side (ms); 51 taps at 5 kHz
REPLAY_TRIPLETS      = True              # also run the prime_triplet post windows (rejection and raw amplitude only;
                                         # online they neither update the normaliser nor finetune the model)
RUN_MODEL            = True              # PRIME model: calibration, predictions, finetuning; sections 7 and 8

# ---------------------------------------------------------------- outputs and support files
RESULTS_ROOT = None              # None: <Loop>/offline_three_arm/sub-<ID>, one folder per arm below it
SUPPORT_DIR  = None              # None: offline_replay_three_arm/ next to this notebook (post_lowpass.py, the patches)

# ---------------------------------------------------------------- decider settings that changed between sessions
SESSION_OVERRIDES = {            # the values the session ran with
    "pilot003": {"ocular_z_threshold": 2.0, "sspsir_timerange": ["automatic", 50]},   # before f4fd395 and 11bf662
    "pilot004": {"ocular_z_threshold": 2.0, "sspsir_timerange": ["automatic", 50]},
    "pilot005": {"sspsir_timerange": ["automatic", 50]},                               # before 11bf662 (2026-10-02)
}

ALLOW_CPU = False                # OnlinePredictor hard-codes CUDA; True runs it on the CPU when there is no GPU

## 1 · Setup

### 1.1 Paths

The Loop folder is found as in `offline_replay.ipynb`: the `LOOP_ROOT` environment variable, `D:\Linus\Loop`, or a
folder above this notebook that holds `BIDS` and `prime`. Both prime layouts work: the decider repository in
`prime/decider` (lab PC) or in `prime` itself, with the forward model in `prime/offline_data`,
`prime/decider/offline_data` or `offline_data`. Results never go into `prime/`.

In [ ]:
import ast, contextlib, hashlib, importlib.abc, importlib.util, inspect, io, json, linecache, logging, os, platform
import re, sys, textwrap, time, types, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
from scipy import signal, stats

warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
})
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)

# conditions: the colours of PRIME_online_performance.ipynb and offline_replay.ipynb
COLOR_PREDETERMINED = "#5b8ff9"
COLOR_PRIME_SINGLE  = "#e8684a"
CONDITION_COLORS = {"predetermined_single": COLOR_PREDETERMINED, "prime_single_pulse": COLOR_PRIME_SINGLE}
# data sources: the online log (grey, dashed, open markers) and the three arms. A, B, C were checked as one
# categorical set: separable with colour-vision deficiencies, at least 3:1 contrast on white.
SOURCE_STYLE = {
    "online": dict(color="#595959", ls="--", marker="o", mfc="white"),
    "A":      dict(color="#199e70", ls="-",  marker="s", mfc="#199e70"),
    "B":      dict(color="#c98500", ls="-",  marker="^", mfc="#c98500"),
    "C":      dict(color="#4a3aa7", ls="-",  marker="D", mfc="#4a3aa7"),
}

if not hasattr(np, "trapz"):     # NumPy >= 2.4 removed np.trapz; Preprocessor.calibrate() still calls it
    np.trapz = np.trapezoid      # (bad-channel AUC). Same function, new name; in memory only.


def _find_loop_root() -> Path:
    env = os.environ.get("LOOP_ROOT")
    candidates = [env] if env else []
    candidates += [r"D:\Linus\Loop", str(Path.home() / "Desktop" / "Loop")]
    for c in candidates:
        if c and (Path(c) / "BIDS").is_dir() and (Path(c) / "prime").is_dir():
            return Path(c)
    for d in (Path.cwd(), *Path.cwd().parents):
        if (d / "BIDS").is_dir() and (d / "prime").is_dir():
            return d
    raise FileNotFoundError("Loop root not found; set the LOOP_ROOT environment variable.")


def first_existing(paths, what):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    raise FileNotFoundError(f"{what} not found; looked for:\n  " + "\n  ".join(str(p) for p in paths))


def git_head(repo: Path) -> str:
    # Commit of the prime repository, read from .git without running git.
    git = repo / ".git"
    try:
        head = (git / "HEAD").read_text().strip()
        if not head.startswith("ref:"):
            return head
        ref = head.split(maxsplit=1)[1]
        if (git / ref).is_file():
            return (git / ref).read_text().strip()
        for line in (git / "packed-refs").read_text().splitlines():
            if line.endswith(" " + ref):
                return line.split()[0]
    except OSError:
        pass
    try:                                                   # last entry of the reflog
        return (git / "logs" / "HEAD").read_text().strip().splitlines()[-1].split()[1]
    except (OSError, IndexError):
        return "unknown"


# ---------------------------------------------------------------- paths, all derived from SUBJECT_ID
LOOP_ROOT   = _find_loop_root()
PRIME_DIR   = LOOP_ROOT / "prime"
DECIDER_DIR = first_existing([d for d in (PRIME_DIR / "decider", PRIME_DIR)
                              if (d / "prime_core" / "preprocessing" / "preprocessor.py").is_file()],
                             f"prime_core in {PRIME_DIR / 'decider'} or {PRIME_DIR}")
FORWARD_PATH = first_existing([d / "offline_data" / "fsaverage" / "fsaverage-fwd.fif"
                               for d in (PRIME_DIR, DECIDER_DIR, LOOP_ROOT)], "forward model fsaverage-fwd.fif")
SUPPORT_DIR = first_existing([p for p in ([Path(SUPPORT_DIR)] if SUPPORT_DIR else
                                          [Path.cwd() / "offline_replay_three_arm",
                                           LOOP_ROOT / "Loop_analysis" / "offline_replay_three_arm",
                                           LOOP_ROOT / "scripts_analysis" / "offline_replay_three_arm"])
                              if (p / "post_lowpass.py").is_file()],
                             "offline_replay_three_arm/ with post_lowpass.py (set SUPPORT_DIR)")
SESSION_DIR = LOOP_ROOT / "BIDS" / f"sub-{SUBJECT_ID}" / SESSION_ID
if RUN_ID is None:
    runs = sorted({p.parent.parent.name for p in SESSION_DIR.glob("*/prime/trials_intervention.csv")})
    if len(runs) != 1:
        raise FileNotFoundError(f"expected one run folder with prime/trials_intervention.csv in {SESSION_DIR}, "
                                f"found {runs}; set RUN_ID")
    RUN_ID = runs[0]
ONLINE_DIR   = SESSION_DIR / RUN_ID / "prime"             # everything the online decider wrote
RESULTS_ROOT = Path(RESULTS_ROOT) if RESULTS_ROOT else LOOP_ROOT / "offline_three_arm" / f"sub-{SUBJECT_ID}"
if not ONLINE_DIR.is_dir():
    raise FileNotFoundError(ONLINE_DIR)
if RESULTS_ROOT.resolve().is_relative_to(PRIME_DIR.resolve()):     # outputs never go into prime/
    raise ValueError(f"output folder inside prime/: {RESULTS_ROOT}")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Loop folder   : {LOOP_ROOT}")
print(f"decider code  : {DECIDER_DIR}")
print(f"forward model : {FORWARD_PATH}")
print(f"support files : {SUPPORT_DIR}")
print(f"session       : {ONLINE_DIR.relative_to(LOOP_ROOT)}")
print(f"results       : {RESULTS_ROOT}")

### 1.2 The online code, with the two patches applied in memory

Both patches are unified diffs. Their hunks are applied to the text of `prime_config.py`, `resampling.py`,
`preprocessor.py` and `dipole_fitter.py` without fuzz: the context lines have to match, and a hunk is placed at
another line only if its lines occur there and nowhere else. An import hook then serves the result under the
original module names. A patched module keeps its real `__file__`, so `configs/prime.yaml` and every other path
relative to it resolves as usual; tracebacks show the patched text. `post_lowpass.py` is served from the support
folder as `prime_core.preprocessing.utils.post_lowpass`. The hunks for `prime.yaml` and `2_intervention.py` are not
needed here: every rate and the low-pass are passed explicitly, and the decider itself is not imported.

A patch is skipped when the code already contains it (`get_post_sfreq` / `get_post_lowpass` in `prime_config.py`).
If `prime/` holds a version the patches were not made for, the first hunk that does not fit stops the notebook and
names its file and line. The table below lists the version of each file and how it was imported.

As in `offline_replay.ipynb`: `sys.dont_write_bytecode = True`; with `ALLOW_CPU` and no GPU, `OnlinePredictor`'s
module sees a `torch` whose `device()` returns the CPU; `SEED` and the model file names are read from
`2_intervention.py` without importing it (it loads the TMS driver).

The kernel must not have imported `prime_core` before this cell. Restart it if it has.

In [ ]:
# ---------------------------------------------------------------- the two patches
PATCHES = [   # (name, file in SUPPORT_DIR, text in prime_config.py when prime/ already contains the patch)
    ("5 kHz post window",      "post_5khz.patch",    "def get_post_sfreq"),
    ("post-stimulus low-pass", "post_lowpass.patch", "def get_post_lowpass"),
]
PATCHED_MODULES = {   # file in the decider repository -> module name
    "prime_core/prime_config.py":                   "prime_core.prime_config",
    "prime_core/preprocessing/utils/resampling.py": "prime_core.preprocessing.utils.resampling",
    "prime_core/preprocessing/preprocessor.py":     "prime_core.preprocessing.preprocessor",
    "prime_core/preprocessing/dipole_fitter.py":    "prime_core.preprocessing.dipole_fitter",
}
NOT_NEEDED_IN_MEMORY = {"prime_core/configs/prime.yaml", "2_intervention.py"}
KNOWN_VERSIONS = {   # sha256 of the file (LF line ends) -> prime commits
    "415de49f9ab7c31b96c5de27231a212533b66af08a2c846a1fd33969315ca322": "ee41369 (post-5khz)",
    "65afe25cad92da30f83eda25f83df5683cb00f688e2ec1bfc4fdd03580368abc": "ee41369 (post-5khz)",
    "813524fa30c4a1269d4bb29414fb306f888f025f7a4f041b8784695f930b8fb2": "ee41369 (post-5khz)",
    "e132e5e444be33aab9f200dcc5a164214184e9e99c40869a24270497f045ca1b": "ee41369 (post-5khz)",
    "272dd11aaadef21211fe68d0739d7bedd105fb752fa6bf017b190f64a24ae336": "ee41369 + low-pass patch",
    "ddd3845ec0bfc0de4c8e5f36d2df05cad2e6d77148070d8bb120c3dba2d848c4": "ee41369 + low-pass patch",
    "215cb597f991bfd8e13dbcd06d061bd1e66557fd7031a46e3a92fbd8a48b1183": "f4fd395 / 11bf662",
    "b15eeaa25cd49f7d7f3c7dfc16d204419202237d4aa6f3c383e21edc607366d3": "f4fd395 / 11bf662",
    "bd7d868b28a5b86f6b099d3a147c8c614e7d3330fd9f94b52d8ec9c607c301cc": "f4fd395 / 11bf662",
    "cf41b35a3d0795bc195ec258736d5fda7f8b704cfdd10c4cbc163969683b908a": "f4fd395 / 11bf662",
}
for _, fname, _ in PATCHES:
    if not (SUPPORT_DIR / fname).is_file():
        raise FileNotFoundError(SUPPORT_DIR / fname)


def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def parse_unified_diff(text):
    # {path: [(old_start, old_lines, new_lines), ...]} from a git diff; hunk lengths are checked.
    files, path, hunk, left = {}, None, None, None
    for line in text.splitlines(keepends=True):
        if line.startswith("diff --git "):
            path = hunk = None
            continue
        if hunk is None:
            if line.startswith("+++ "):
                p = line[4:].strip()
                path = p[2:] if p.startswith("b/") else p
                files.setdefault(path, [])
            elif line.startswith("@@ ") and path is not None:
                m = re.match(r"@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@", line)
                hunk = (int(m.group(1)), [], [])
                left = [int(m.group(2) or 1), int(m.group(4) or 1)]
                files[path].append(hunk)
            continue
        if line.startswith("\\"):                     # "\ No newline at end of file"
            continue
        tag, body = (" ", "\n") if line in ("\n", "\r\n") else (line[:1], line[1:])
        if tag in " -":
            hunk[1].append(body)
            left[0] -= 1
        if tag in " +":
            hunk[2].append(body)
            left[1] -= 1
        if tag not in " -+":
            raise ValueError(f"unexpected line in a hunk of {path}: {line!r}")
        if left == [0, 0]:
            hunk = None
    if hunk is not None:
        raise ValueError(f"the last hunk of {path} is incomplete: the patch file is truncated")
    return files


def apply_hunks(text, hunks, label):
    # Applies the hunks exactly. Returns the new text and a note for each hunk that sat at another line.
    lines = text.splitlines(keepends=True)
    if lines and not lines[-1].endswith("\n"):
        lines[-1] += "\n"
    out, pos, notes = [], 0, []
    for k, (start, old, new) in enumerate(hunks, 1):
        i = start - 1
        if lines[i:i + len(old)] != old:
            hits = [j for j in range(pos, len(lines) - len(old) + 1) if lines[j:j + len(old)] == old]
            if len(hits) != 1:
                raise ValueError(f"{label}: hunk {k} (line {start}) does not fit the file on disk "
                                 f"({len(hits)} matches) -- this prime version is not the one the patch was made for")
            notes.append(f"hunk {k} at line {hits[0] + 1} instead of {start}")
            i = hits[0]
        out += lines[pos:i] + new
        pos = i + len(old)
    return "".join(out + lines[pos:]), notes


class InMemorySource(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    # Serves the listed modules from source text. __file__ stays the real file; the code object is named
    # "<file> <tag>" and registered in linecache, so tracebacks show the patched lines.
    def __init__(self):
        self.modules = {}

    def add(self, fullname, file_path, source, tag):
        code_name = f"{file_path} <{tag}>"
        self.modules[fullname] = (str(file_path), source, code_name)
        linecache.cache[code_name] = (len(source), None, source.splitlines(True), code_name)

    def find_spec(self, fullname, path=None, target=None):
        if fullname in self.modules:
            return importlib.util.spec_from_file_location(fullname, self.modules[fullname][0], loader=self)
        return None

    def create_module(self, spec):
        return None

    def exec_module(self, module):
        file_path, source, code_name = self.modules[module.__name__]
        module.__in_memory_patch__ = code_name
        exec(compile(source, code_name, "exec", dont_inherit=True), module.__dict__)


if "CODE_PROVENANCE" not in globals():                    # once per kernel
    loaded = sorted(m for m in sys.modules if m == "prime_core" or m.startswith("prime_core."))
    if loaded:
        raise RuntimeError(f"{loaded[0]} was imported before the patches: restart the kernel and run from the top")
    ORIGINAL = {rel: (DECIDER_DIR / rel).read_text(encoding="utf-8") for rel in PATCHED_MODULES}
    _sources, PATCH_LOG = dict(ORIGINAL), []
    for name, fname, marker in PATCHES:
        text = (SUPPORT_DIR / fname).read_text(encoding="utf-8")
        entry = dict(patch=name, file=fname, sha256=sha256(text)[:12])
        if marker in _sources["prime_core/prime_config.py"]:
            PATCH_LOG.append({**entry, "status": "already in prime/", "notes": ""})
            continue
        hunks = parse_unified_diff(text)
        unknown = sorted(set(hunks) - set(PATCHED_MODULES) - NOT_NEEDED_IN_MEMORY)
        if unknown:
            raise ValueError(f"{fname} changes files this notebook does not patch in memory: {unknown}")
        notes = []
        for rel, file_hunks in hunks.items():
            if rel in PATCHED_MODULES:
                _sources[rel], moved = apply_hunks(_sources[rel], file_hunks, f"{fname}, {rel}")
                notes += [f"{Path(rel).name}: {m}" for m in moved]
        PATCH_LOG.append({**entry, "status": "applied in memory", "notes": "; ".join(notes)})
    HOOK = InMemorySource()
    for rel, module in PATCHED_MODULES.items():
        if _sources[rel] != ORIGINAL[rel]:
            HOOK.add(module, DECIDER_DIR / rel, _sources[rel], "patched in memory by offline_replay_three_arm")
    if not (DECIDER_DIR / "prime_core" / "preprocessing" / "utils" / "post_lowpass.py").is_file():
        HOOK.add("prime_core.preprocessing.utils.post_lowpass", SUPPORT_DIR / "post_lowpass.py",
                 (SUPPORT_DIR / "post_lowpass.py").read_text(encoding="utf-8"), "offline_replay_three_arm")
    sys.meta_path.insert(0, HOOK)
    CODE_PROVENANCE = dict(prime_commit=git_head(DECIDER_DIR), decider_dir=str(DECIDER_DIR),
                           support_dir=str(SUPPORT_DIR), patches=PATCH_LOG,
                           files_on_disk={rel: sha256(t) for rel, t in ORIGINAL.items()})

# ---------------------------------------------------------------- the online code, read-only
sys.dont_write_bytecode = True                            # no __pycache__ written into prime/
if str(DECIDER_DIR) not in sys.path:
    sys.path.insert(0, str(DECIDER_DIR))
import mne
try:
    import mne_icalabel
except ImportError as err:
    raise ImportError("the calibration replay needs mne-icalabel: pip install mne-icalabel") from err
import prime_core.preprocessing.preprocessor as preprocessor_module
from prime_core.preprocessing.preprocessor import (
    Preprocessor, crop_eeg_buffer, _butter_filter, _interpolate_bad_channels,
    _detect_bad_channels_qc, _detect_bad_channels_post,
)
from prime_core.preprocessing.dipole_fitter import DipoleFitter
from prime_core.preprocessing.config import get_default_config
from prime_core.preprocessing.utils.mad import global_mad_zscore_rejected, local_mad_zscore_rejected
from prime_core.preprocessing.utils.post_lowpass import PostLowpass
from prime_core.tep_normalizer import TEPNormalizer
from prime_core.prime_config import (
    get_raw_sfreq, get_processed_sfreq, get_calibration_time_range, get_qc_time_range,
    get_model_time_range, get_post_initial_time_range, get_post_time_range, get_dipole_time_range,
)
mne.set_log_level("ERROR")

for params, func in ((("ica_sfreq", "post_sfreq", "post_lowpass"), Preprocessor.__init__),
                     (("sfreq",), DipoleFitter.__init__)):
    missing = [p for p in params if p not in inspect.signature(func).parameters]
    if missing:
        raise RuntimeError(f"{func.__qualname__} has no {missing}: the patches did not reach the imported code")

rows = []
for rel, module in PATCHED_MODULES.items():
    digest = CODE_PROVENANCE["files_on_disk"][rel]
    rows.append({"module": module, "sha256 on disk": digest[:12],
                 "prime version": KNOWN_VERSIONS.get(digest, "not one of the known versions"),
                 "imported": "patched in memory" if getattr(sys.modules[module], "__in_memory_patch__", None)
                             else "as on disk"})
_lp = sys.modules["prime_core.preprocessing.utils.post_lowpass"]
rows.append({"module": _lp.__name__, "sha256 on disk": sha256(Path(_lp.__file__).read_text(encoding="utf-8"))[:12],
             "prime version": "–", "imported": f"from {Path(_lp.__file__).parent.name}/"})
display(pd.DataFrame(CODE_PROVENANCE["patches"]).set_index("patch"))
display(pd.DataFrame(rows).set_index("module"))

# ---------------------------------------------------------------- constants of the decider (2_intervention.py, not imported)
DECIDER_SRC = (DECIDER_DIR / "2_intervention.py").read_text(encoding="utf-8")


def decider_constant(name):
    for node in ast.parse(DECIDER_SRC).body:
        if isinstance(node, ast.Assign) and any(getattr(t, "id", None) == name for t in node.targets):
            try:
                return ast.literal_eval(node.value)
            except ValueError:
                return ast.get_source_segment(DECIDER_SRC, node.value)
    raise KeyError(f"{name} not found in 2_intervention.py")


SEED         = decider_constant("SEED")                        # seed of the online OnlinePredictor
BLOCK_TRIALS = decider_constant("INTERVENTION_BLOCK_TRIALS")   # intervention trials per block (checked in 1.3)
PRETRAINED_PATH   = DECIDER_DIR / "offline_results" / "train" / "pretrained_all.pt"
BACKROTATION_PATH = DECIDER_DIR / "offline_results" / "train" / "global_backrotation_all.npy"
for const, path in (("PRETRAINED_MODEL_PATH", PRETRAINED_PATH), ("GLOBAL_BACKROTATION_PATH", BACKROTATION_PATH)):
    if path.name not in decider_constant(const):
        raise ValueError(f"2_intervention.py: {const} = {decider_constant(const)}, not {path.name}")

# ---------------------------------------------------------------- the PRIME model: GPU, or the CPU fallback
if RUN_MODEL:
    import torch
    import prime_core.online_predictor as online_predictor_module
    from prime_core.online_predictor import OnlinePredictor
    logging.getLogger("prime_core.models.s4").setLevel(logging.WARNING)   # "Constructing S4 ..." messages
    if torch.cuda.is_available():
        DEVICE_NOTE = f"CUDA, {torch.cuda.get_device_name(0)}"
    elif ALLOW_CPU:
        class _CPUTorch(types.ModuleType):
            # OnlinePredictor calls torch.device("cuda"); this answers with the CPU. Only its module sees it.
            def __getattr__(self, name):
                return getattr(torch, name)

            @staticmethod
            def device(*args, **kwargs):
                return torch.device("cpu")

        online_predictor_module.torch = _CPUTorch("torch")
        DEVICE_NOTE = "CPU (ALLOW_CPU; predictions differ from a GPU run in the last digits)"
    else:
        raise RuntimeError("OnlinePredictor needs CUDA: run on the GPU machine, set ALLOW_CPU = True, "
                           "or RUN_MODEL = False")
else:
    DEVICE_NOTE = "PRIME model not run (RUN_MODEL = False)"

# ---------------------------------------------------------------- time windows (configs/prime.yaml)
RAW_SFREQ, PROC_SFREQ    = get_raw_sfreq(), get_processed_sfreq()   # 5000 / 1000 Hz
CAL_TMIN, CAL_TMAX       = get_calibration_time_range()             # _trial_raw.npy [-1100, 100] ms
QC_TMIN, QC_TMAX         = get_qc_time_range()                      # _pre_raw.npy   [-210, -10] ms
POST_I_TMIN, POST_I_TMAX = get_post_initial_time_range()            # _post_raw.npy  [-30, 100] ms
POST_TMIN, POST_TMAX     = get_post_time_range()                    # processed post epoch [1, 100] ms
DIP_TMIN, DIP_TMAX       = get_dipole_time_range()                  # dipole window [38, 50] ms
MODEL_TMIN, MODEL_TMAX   = get_model_time_range()                   # model window [-65, -15] ms
DEFAULT_CFG              = get_default_config()
OVERRIDES                = SESSION_OVERRIDES.get(SUBJECT_ID, {})
unknown = set(OVERRIDES) - {"ocular_z_threshold", "sspsir_timerange"}
if unknown:
    raise ValueError(f"SESSION_OVERRIDES[{SUBJECT_ID!r}]: unknown setting(s) {sorted(unknown)}")
ms = lambda w: f"[{w[0] * 1e3:.0f}, {w[1] * 1e3:.0f}] ms"

print(f"prime commit    : {CODE_PROVENANCE['prime_commit'][:10]}, SEED {SEED}")
print(f"windows         : baseline {ms(DEFAULT_CFG.baseline)}, first blank {ms(DEFAULT_CFG.artifact_window_1)}, "
      f"post epoch {ms((POST_TMIN, POST_TMAX))}, dipole {ms((DIP_TMIN, DIP_TMAX))}")
print(f"settings here   : ocular z {DEFAULT_CFG.trial_reject_opts.ocular.z_threshold}, "
      f"SSP-SIR timerange {list(DEFAULT_CFG.ssp_sir_opts.timerange)}")
print(f"overrides       : {OVERRIDES or 'none'}")
print(f"device          : {DEVICE_NOTE}")
print(f"versions        : python {platform.python_version()}, numpy {np.__version__}, scipy {scipy.__version__}, "
      f"mne {mne.__version__}, mne-icalabel {mne_icalabel.__version__}"
      + (f", torch {torch.__version__}" if RUN_MODEL else ""))

### 1.3 The online session

`trials_intervention.csv` defines the replay: the calibration trials and the intervention trials in session order.
`g` is the intervention trial index the decider counts; the online checkpoints are named after it
(`classifier_after_trial_<g>.pt`). `lat_ms` is the trigger-to-pulse latency of PRIME trials. The evaluation tables
list the open-loop trials of section 8 (the baseline block is not used). The post-stimulus rate the session ran at
is read from its calibration bundle (stamped since the 5 kHz patch, otherwise from the length of the SSP-SIR kernel);
it decides which arm is the control.

In [ ]:
TRIALS = pd.read_csv(ONLINE_DIR / "trials_intervention.csv")
TRIALS["stem"] = TRIALS.stage + "_" + TRIALS.trial_in_stage.map("{:04d}".format)
TRIALS["trial_order"] = np.arange(len(TRIALS))
TRIALS["lat_ms"] = ((TRIALS.pulse_time - TRIALS.trigger_time) * 1e3).round(1)   # PRIME trials: trigger -> pulse

CAL = TRIALS[TRIALS.stage == "calibration"].reset_index(drop=True)
IV = TRIALS[TRIALS.stage.str.startswith("intervention_block_")].reset_index(drop=True)
IV["g"] = np.arange(len(IV))                          # intervention trial index, as the decider counts it
SINGLES = IV[IV.condition != "prime_triplet"].reset_index(drop=True)
TRIPLETS = IV[IV.condition == "prime_triplet"].reset_index(drop=True)
BLOCKS = list(dict.fromkeys(IV.stage))
# Online name of the checkpoint saved after a block's last trial: classifier_after_trial_<trials done>.pt
BLOCK_END = {block: int(IV.g[IV.stage == block].max()) + 1 for block in BLOCKS}
for block, n in IV.stage.value_counts().items():
    if n != BLOCK_TRIALS:
        print(f"note: {block} has {n} trials, the decider's block size is {BLOCK_TRIALS}")

OPEN_TRIALS = []
for block, name in [(f"evaluation_{k}", f"trials_evaluation_{k}.csv") for k in range(1, 5)]:   # baseline block not used
    if (ONLINE_DIR / name).is_file():
        d = pd.read_csv(ONLINE_DIR / name)
        prefix = f"evaluation_{int(d.run_index.iloc[0])}"
        OPEN_TRIALS.append(pd.DataFrame({"block": block,
                                         "stem": prefix + "_" + d.trial_in_stage.map("{:04d}".format)}))
OPEN_TRIALS = pd.concat(OPEN_TRIALS, ignore_index=True) if OPEN_TRIALS else pd.DataFrame(columns=["block", "stem"])

BUNDLE_PATH = next((ONLINE_DIR / n for n in ("calibration_bundle.npy", "calibration_bunble.npy")
                    if (ONLINE_DIR / n).is_file()), ONLINE_DIR / "calibration_bundle.npy")
ONLINE = dict(label="online log")
if BUNDLE_PATH.is_file():
    ONLINE["bundle"] = np.load(BUNDLE_PATH, allow_pickle=True).item()
if (ONLINE_DIR / "dipole_fitting_info.npy").is_file():
    ONLINE["fitting_info"] = np.load(ONLINE_DIR / "dipole_fitting_info.npy", allow_pickle=True).item()
if (ONLINE_DIR / "calibration_summary.csv").is_file():
    ONLINE["n_cal_summary"] = int(pd.read_csv(ONLINE_DIR / "calibration_summary.csv").num_calibration_trials.iloc[0])

print(f"{len(CAL)} calibration + {len(IV)} intervention trials ({', '.join(BLOCKS)}); "
      f"{len(OPEN_TRIALS)} open-loop trials ({', '.join(OPEN_TRIALS.block.unique()) or 'none'})")
display(pd.crosstab(IV.condition, IV.is_forced.map({False: "not forced", True: "forced"}), margins=True))

### 1.4 Completeness check

Required inputs stop the notebook when incomplete; optional ones reduce what can be compared. The `_pre_raw`
buffers are needed for the model (prediction windows); the online checkpoints only for the online source in
sections 2.4 and 8; the open-loop buffers only for section 8. Then the control arm is set: the arm that processes
the post window as the session did.

In [ ]:
REPLAYED = IV if REPLAY_TRIPLETS else SINGLES
ITEMS = [   # (input, files, required)
    ("forward model (fsaverage)", [FORWARD_PATH], True),
    ("online calibration bundle", [BUNDLE_PATH], True),
    ("online dipole_fitting_info.npy", [ONLINE_DIR / "dipole_fitting_info.npy"], True),
    ("calibration _trial_raw.npy (ICA window)", [ONLINE_DIR / f"{s}_trial_raw.npy" for s in CAL.stem], True),
    ("raw post-buffers _post_raw.npy, calibration", [ONLINE_DIR / f"{s}_post_raw.npy" for s in CAL.stem], True),
    ("raw post-buffers _post_raw.npy, single pulses", [ONLINE_DIR / f"{s}_post_raw.npy" for s in SINGLES.stem], True),
    ("raw post-buffers _post_raw.npy, triplets", [ONLINE_DIR / f"{s}_post_raw.npy" for s in TRIPLETS.stem],
     REPLAY_TRIPLETS),
    ("pretrained model", [PRETRAINED_PATH], RUN_MODEL),
    ("global back-rotation", [BACKROTATION_PATH], RUN_MODEL),
    ("raw pre-buffers _pre_raw.npy (model input)",
     [ONLINE_DIR / f"{s}_pre_raw.npy" for s in pd.concat([CAL.stem, SINGLES.stem])], RUN_MODEL),
    ("online checkpoints: after calibration and block ends",
     [ONLINE_DIR / "classifier_after_calibration.pt"]
     + [ONLINE_DIR / f"classifier_after_trial_{BLOCK_END[b]}.pt" for b in BLOCKS], False),
    ("open-loop _pre_raw + _post_raw (section 8)",
     [ONLINE_DIR / f"{s}_{kind}_raw.npy" for s in OPEN_TRIALS.stem for kind in ("pre", "post")], False),
]
rows, missing = [], []
for what, paths, required in ITEMS:
    found = sum(Path(p).is_file() for p in paths)
    rows.append({"input": what, "expected": len(paths), "found": found, "required": "yes" if required else "",
                 "status": "ok" if found == len(paths) else ("MISSING" if required else "incomplete")})
    if required and found < len(paths):
        missing.append(what)
display(pd.DataFrame(rows).set_index("input"))
if missing:
    raise FileNotFoundError(f"required inputs incomplete: {missing}")

# ---------------------------------------------------------------- the control arm
ONLINE_POST_SFREQ = Preprocessor.bundle_post_sfreq(ONLINE["bundle"])
ONLINE_LOWPASS = PostLowpass.from_bundle(ONLINE["bundle"])
if abs(ONLINE_POST_SFREQ - 1000.0) < 1e-6:
    CONTROL = "A"
elif abs(ONLINE_POST_SFREQ - 5000.0) < 1e-6:
    CONTROL = "C" if ONLINE_LOWPASS is not None else "B"
else:
    CONTROL = None
print(f"online post window: {ONLINE_POST_SFREQ:g} Hz, "
      f"{ONLINE_LOWPASS.describe() if ONLINE_LOWPASS is not None else 'no post-stimulus low-pass'}")
if CONTROL == "C" and (ONLINE_LOWPASS.cutoff_hz, ONLINE_LOWPASS.halfwidth_ms) != (LOWPASS_HZ, LOWPASS_HALFWIDTH_MS):
    print(f"note: the session's low-pass differs from arm C ({LOWPASS_HZ:g} Hz, {LOWPASS_HALFWIDTH_MS:g} ms)")
print(f"control arm: {CONTROL}" + ("" if CONTROL in ARMS_TO_RUN else " -- not in ARMS_TO_RUN, so nothing is "
                                   "checked against the online log"))

## 2 · Calibration replay

### 2.1 Re-run `run_calibration()` in each arm

The calibration is recomputed from scratch, as `run_calibration()` in `2_intervention.py` does it:

`add_trial()` × 125 → `Preprocessor.calibrate()` → `DipoleFitter.calibrate()` → `TEPNormalizer.calibrate()` →
`OnlinePredictor.calibrate()` → `warm_up()`

The input is `_trial_raw.npy` ([−1100, 100] ms). Each arm builds its `Preprocessor` with `ica_sfreq = 1000`, its own
`post_sfreq` and, in arm C, the low-pass settings: `calibrate()` designs the filter, applies it to every calibration
epoch right after the first blank and freezes it in the bundle (`calibration_params["post_lowpass"]`), and every
later pulse uses that frozen filter. The `DipoleFitter` runs at the arm's post rate; its sub-window limits are 3 and
6 ms at any rate (3–6 samples at 1 kHz, 15–30 at 5 kHz). What can differ between the arms:

- **bad channels**: the post-stimulus detector scores the evoked response over [20, 60] ms, now with a different
  bandwidth. A new set changes the interpolation, the ICA input and the model input.
- **ocular gate**: mean and SD of |IC source| over the post epoch, computed on the arm's bandwidth.
- **SOUND, SSP-SIR and the post-stimulus MAD statistics**, all fitted on the calibration epochs. SSP-SIR estimates
  the muscle artifact from the power above 100 Hz, which reaches 2.5 kHz in arm B: the largest expected difference.
- **the surviving trials**, and with them the pre-stimulus MAD statistics, the dipole, the normaliser and the model.

The ICA is fitted on the 1 kHz ICA window with a fixed seed, so it only changes if the bad channels do. Each arm gets
a fresh `OnlinePredictor` built as in `Decider.__init__` (pretrained model, global back-rotation, `SEED`); its
random state after calibration is restored in section 3, so finetuning continues the random stream of its own
calibration, as online. The evoked response handed to SSP-SIR is recorded on the way (for the check in 2.3); the
computation itself is unchanged.

In [ ]:
ICA_SFREQ = 1000.0      # every arm: the ICA window stays at 1 kHz, as in the 5 kHz patch
ARM_SPECS = {
    "A": dict(post_sfreq=1000.0, lowpass=None, folder="arm_A_1kHz", what="1 kHz"),
    "B": dict(post_sfreq=5000.0, lowpass=None, folder="arm_B_5kHz", what="5 kHz"),
    "C": dict(post_sfreq=5000.0, lowpass=(float(LOWPASS_HZ), float(LOWPASS_HALFWIDTH_MS)),
              folder="arm_C_5kHz_lowpass", what=f"5 kHz + {LOWPASS_HZ:g} Hz low-pass"),
}
if not ARMS_TO_RUN or any(a not in ARM_SPECS for a in ARMS_TO_RUN):
    raise ValueError(f"ARMS_TO_RUN must be a subset of {list(ARM_SPECS)}, got {ARMS_TO_RUN}")
ARMS = {}
for name in ARM_SPECS:                       # always in the order A, B, C
    if name in ARMS_TO_RUN:
        arm = dict(ARM_SPECS[name], name=name)
        arm["label"] = f"{name} · {arm['what']}"
        arm["dir"] = RESULTS_ROOT / arm["folder"]
        arm["dir"].mkdir(parents=True, exist_ok=True)
        ARMS[name] = arm
        print(f"{arm['label']:30s}{' (control)' if name == CONTROL else '':11s} -> {arm['dir']}")
REF = CONTROL if CONTROL in ARMS else next(iter(ARMS))    # the arm the others are compared with in sections 4-6

if RUN_MODEL:
    GLOBAL_BACKROTATION = np.load(BACKROTATION_PATH)


def apply_overrides(pp):
    # The decider settings the session ran with (SESSION_OVERRIDES), on the Preprocessor's own config.
    if "ocular_z_threshold" in OVERRIDES:
        pp._cfg.trial_reject_opts.ocular.z_threshold = OVERRIDES["ocular_z_threshold"]
    if "sspsir_timerange" in OVERRIDES:
        pp._cfg.ssp_sir_opts.timerange = list(OVERRIDES["sspsir_timerange"])
    pp._opts = pp._cfg.to_dicts()                      # calibrate() reads this copy, made in __init__
    return pp


def make_preprocessor(arm):
    # The online Preprocessor at the arm's post rate. The low-pass settings are only read by calibrate().
    pp = Preprocessor(FORWARD_PATH, ica_sfreq=ICA_SFREQ, post_sfreq=arm["post_sfreq"],
                      post_lowpass=arm["lowpass"] or False)
    pp.keep_full_epochs = True                         # as Decider.__init__
    return apply_overrides(pp)


def new_predictor():
    # As Decider.__init__: pretrained model, global back-rotation, SEED. The constructor re-seeds numpy and torch.
    return OnlinePredictor(GLOBAL_BACKROTATION, model_path=PRETRAINED_PATH, seed=SEED)


def rng_state():
    return dict(np=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)


def set_rng_state(state):
    np.random.set_state(state["np"])
    torch.set_rng_state(state["torch"])
    if state["cuda"] is not None:
        torch.cuda.set_rng_state_all(state["cuda"])


def trial_times(tmin, n):
    # Sample times of a raw buffer starting at tmin. The decider's timestamps were not saved; crop_eeg_buffer()
    # only reads the first one, so a uniform grid reproduces every crop.
    return tmin + np.arange(n) / RAW_SFREQ


@contextlib.contextmanager
def recording_sspsir():
    # Records the evoked response SSP-SIR is fitted on; the computation is unchanged.
    calls, original = [], preprocessor_module.ssp_sir_to_average

    def recording(data, L, fs, timerange, *args, **kwargs):
        calls.append(dict(data=np.array(data, copy=True), fs=fs, timerange=list(timerange), kwargs=dict(kwargs)))
        return original(data, L, fs, timerange, *args, **kwargs)

    preprocessor_module.ssp_sir_to_average = recording
    try:
        yield calls
    finally:
        preprocessor_module.ssp_sir_to_average = original


def replay_calibration(arm):
    # run_calibration() of 2_intervention.py on the saved calibration trials, in the online order.
    t0 = time.time()
    predictor = new_predictor() if RUN_MODEL else None      # built before the preprocessor, as in Decider.__init__
    pp = make_preprocessor(arm)
    for stem in CAL.stem:
        x = np.load(ONLINE_DIR / f"{stem}_trial_raw.npy")
        pp.add_trial(x, trial_times(CAL_TMIN, len(x)))     # process_pulse(), calibration stage
    log, t1 = io.StringIO(), time.time()
    with contextlib.redirect_stdout(log), recording_sspsir() as sspsir_calls:
        model_buffers, dipole_buffers = pp.calibrate()
        fitter = DipoleFitter(FORWARD_PATH, sfreq=pp.post_sfreq)
        amplitudes = fitter.calibrate(dipole_buffers)
        normalizer = TEPNormalizer()
        labels = normalizer.calibrate(amplitudes)
        if RUN_MODEL:
            predictor.calibrate(model_buffers, labels)
            predictor.warm_up()
    seconds_run_calibration = time.time() - t1
    return dict(pp=pp, fitter=fitter, normalizer=normalizer, predictor=predictor,
                bundle=pp.calibration_params, fitting_info=fitter.fitting_info, lowpass=pp.post_lowpass,
                kept=[int(k) for k in pp.calibration_params["trials_kept"]],
                model_buffers=model_buffers, dipole_buffers=dipole_buffers,
                amplitudes=np.asarray(amplitudes), labels=np.asarray(labels), sspsir=sspsir_calls,
                log=log.getvalue(), rng=rng_state() if RUN_MODEL else None,
                seconds=time.time() - t0, seconds_run_calibration=seconds_run_calibration)


CALIB = {}
for name, arm in ARMS.items():
    CALIB[name] = cal = replay_calibration(arm)
    lp = cal["lowpass"]
    print(f"\n{arm['label']}: {cal['seconds']:.0f} s in all, {cal['seconds_run_calibration']:.1f} s for run_calibration(); "
          f"{len(cal['kept'])} of {len(CAL)} trials used; {lp.describe() if lp is not None else 'no post-stimulus low-pass'}")
    print(textwrap.indent(cal["log"].strip(), "    "))

### 2.2 The online calibration, where it was not saved

The online run saved the bundle and the dipole, but not the calibration amplitudes, the normaliser, the calibration
TEP or the model inputs. They are rebuilt from the stored bundle as in `offline_replay.ipynb`: the bundle goes through
`Preprocessor.from_bundle()`, which takes the post rate and the low-pass from the bundle itself, never from the
config; each surviving calibration trial's `_post_raw` buffer goes through `preprocess_post()`, its [38, 50] ms
window through `fit_trial()` with the online dipole, and `TEPNormalizer.calibrate()` runs on those amplitudes. The
same per-trial route is checked against each arm's batch calibration.

In [ ]:
GATE_NAMES = {"ocular": "ocular", "qc": "pre-stimulus MAD", "post": "post-stimulus MAD"}


def dip_slice(sfreq):
    # [38, 50] ms inside the [1, 100] ms processed post epoch at sfreq (13 samples at 1 kHz, 61 at 5 kHz)
    return slice(int(round((DIP_TMIN - POST_TMIN) * sfreq)), int(round((DIP_TMAX - POST_TMIN) * sfreq)) + 1)


def post_raw(stem):
    # The online _post_raw.npy [-30, 100] ms and its sample times; preprocess_post() crops and resamples it itself.
    x = np.load(ONLINE_DIR / f"{stem}_post_raw.npy")
    return x, trial_times(POST_I_TMIN, len(x))


def model_crop(pre_epoch):
    # The [-65, -15] ms model window out of the [-210, -10] ms pre epoch (last step of preprocess_pre)
    model, _ = crop_eeg_buffer(pre_epoch.T, np.asarray([QC_TMIN]), MODEL_TMIN, MODEL_TMAX, sfreq=PROC_SFREQ)
    return np.ascontiguousarray(model.T, dtype=np.float64)


def run_preprocess_pre(pp, stem):
    # preprocess_pre() on the pulse-aligned _pre_raw window [-210, -10] ms, as preprocess_pre_from_pulse() in
    # 2_intervention.py. Returns the model window (None if a gate rejected it) and the epoch before the gates.
    x = np.load(ONLINE_DIR / f"{stem}_pre_raw.npy")
    pre = pp.preprocess_pre(x, trial_times(QC_TMIN, len(x)), from_pulse=True)
    return pre, pp.last_pre_full


def calibration_epochs(pp, fitter, kept):
    # Surviving calibration trials through preprocess_post() one at a time: the [1, 100] ms epoch as captured
    # before the reject gates (last_post_full) and the dipole amplitude of its [38, 50] ms window.
    epochs, amps, idx = [], [], dip_slice(pp.post_sfreq)
    for k in kept:
        pp.preprocess_post(*post_raw(CAL.stem[k]))
        epochs.append(pp.last_post_full)
        amps.append(fitter.fit_trial(pp.last_post_full[:, idx]))
    return np.array(epochs), np.array(amps)


# ---- online: the saved state, plus what was not saved
ONLINE["pp"] = apply_overrides(Preprocessor.from_bundle(ONLINE["bundle"], FORWARD_PATH))
ONLINE["pp"].keep_full_epochs = True
ONLINE["lowpass"] = ONLINE["pp"].post_lowpass
ONLINE["fitter"] = DipoleFitter(FORWARD_PATH, sfreq=ONLINE["pp"].post_sfreq)
ONLINE["fitter"]._fitting_info = ONLINE["fitting_info"]          # DipoleFitter has no public restore method
ONLINE["kept"] = [int(k) for k in ONLINE["bundle"]["trials_kept"]]
ONLINE["cal_epochs"], ONLINE["amplitudes"] = calibration_epochs(ONLINE["pp"], ONLINE["fitter"], ONLINE["kept"])
ONLINE["dipole_buffers"] = ONLINE["cal_epochs"][:, :, dip_slice(ONLINE["pp"].post_sfreq)]
ONLINE["normalizer"] = TEPNormalizer()
ONLINE["labels"] = np.asarray(ONLINE["normalizer"].calibrate(ONLINE["amplitudes"]))
if RUN_MODEL:
    ONLINE["model_buffers"] = np.array([model_crop(run_preprocess_pre(ONLINE["pp"], CAL.stem[k])[1])
                                        for k in ONLINE["kept"]])
print(f"online: {len(ONLINE['kept'])} surviving calibration trials rebuilt at {ONLINE['pp'].post_sfreq:g} Hz "
      f"(calibration_summary.csv: {ONLINE.get('n_cal_summary', 'n/a')})")

# ---- arms: the same per-trial epochs (for the calibration TEP), checked against the batch calibration
for name, cal in CALIB.items():
    cal["cal_epochs"], amps = calibration_epochs(cal["pp"], cal["fitter"], cal["kept"])
    print(f"{ARMS[name]['label']}: per-trial vs batch calibration amplitudes, max |diff| "
          f"{np.max(np.abs(amps - cal['amplitudes'])):.1e} µAm")

### 2.3 Does the control arm reproduce the online calibration?

The control arm processes the post window as the session did, so its calibration should equal the online one: the
same surviving trials, bad channels, ICA, SOUND and SSP-SIR operators and dipole, up to float noise. If it does not,
the session most likely ran with a decider setting that has changed since; the two that have changed in the history
of `prime` are the ocular z threshold (2.0 → 2.5 in `f4fd395`) and the SSP-SIR window (50 → 40 ms in `11bf662`). For
the SSP-SIR window this cell finds the value directly: when everything up to SOUND matches but the SSP-SIR kernel
does not, it recomputes the kernel from the recorded evoked response for several window lengths and reports the one
that matches the online kernel. Set the reported value in `SESSION_OVERRIDES` and run the notebook again.

The three-arm comparison stays valid without a reproduced control (all arms use the same settings), but its
numbers then describe this decider version rather than the session.

In [ ]:
def rel_diff(a, b):
    return float(np.linalg.norm(np.asarray(a) - np.asarray(b)) / np.linalg.norm(np.asarray(b)))


def same_array(a, b, rtol=1e-5, atol=1e-8):
    a, b = np.asarray(a), np.asarray(b)
    return a.shape == b.shape and bool(np.allclose(a, b, rtol=rtol, atol=atol))


def n_sspsir(bundle):
    P = bundle["sspsir_suppression_matrix_P"]
    return int(round(np.trace(np.eye(len(P)) - P)))


def reproduction_table(cal):
    b, on = cal["bundle"], ONLINE["bundle"]
    rows = []

    def add(item, same, detail):
        rows.append({"outcome": item, "reproduced": "yes" if same else "NO", "replay vs online": detail})

    ints = lambda v: sorted(int(i) for i in v)
    add("calibration trials kept", cal["kept"] == ONLINE["kept"], f"{len(cal['kept'])} vs {len(ONLINE['kept'])}")
    for gate, label in GATE_NAMES.items():
        x, y = ints(b["trials_dropped"][gate]), ints(on["trials_dropped"][gate])
        add(f"dropped by {label}", x == y, f"{len(x)} vs {len(y)}")
    x, y = sorted(map(str, b["bad_channels"])), sorted(map(str, on["bad_channels"]))
    add("bad channels", x == y, f"{', '.join(x) or '–'} vs {', '.join(y) or '–'}")
    ua, uo = b["ica"].unmixing_matrix_, on["ica"].unmixing_matrix_
    add("ICA unmixing matrix", same_array(ua, uo),
        f"max |Δ| {np.max(np.abs(ua - uo)):.1e}" if ua.shape == uo.shape else "different size")
    xa, xo = [int(i) for i in b["ica"].exclude], [int(i) for i in on["ica"].exclude]
    add("ICA excluded components", xa == xo, f"{xa} vs {xo}")
    add("SOUND filter", same_array(b["sound_filter"], on["sound_filter"]),
        f"rel. diff {rel_diff(b['sound_filter'], on['sound_filter']):.1e}"
        if np.shape(b["sound_filter"]) == np.shape(on["sound_filter"]) else "different size")
    ka, ko = b["sspsir_filter_kernel"], on["sspsir_filter_kernel"]
    add("SSP-SIR artifact kernel", same_array(ka, ko),
        f"rel. diff {rel_diff(ka, ko):.1e}" if np.shape(ka) == np.shape(ko) else "different length")
    add("SSP-SIR components removed", n_sspsir(b) == n_sspsir(on), f"{n_sspsir(b)} vs {n_sspsir(on)}")
    fi, fo = cal["fitting_info"], ONLINE["fitting_info"]
    add("dipole vertex", int(fi["position_index"]) == int(fo["position_index"]),
        f"{int(fi['position_index'])} vs {int(fo['position_index'])}")
    ra, ro = tuple(int(v) for v in fi["index_range"]), tuple(int(v) for v in fo["index_range"])
    add("dipole window (samples)", ra == ro, f"{ra} vs {ro}")
    if cal["kept"] == ONLINE["kept"]:
        d = float(np.max(np.abs(cal["amplitudes"] / ONLINE["amplitudes"] - 1)))
        add("calibration amplitudes", d < 1e-3, f"max relative difference {d:.1e}")
    return pd.DataFrame(rows).set_index("outcome")


def sspsir_window_probe(cal, windows=(20, 30, 40, 50, 60, 70, 80)):
    # The SSP-SIR kernel for other window lengths, on the evoked response the control arm fitted SSP-SIR to.
    call = cal["sspsir"][0]
    L = cal["pp"]._forward["sol"]["data"]
    L = L - L.mean(axis=0)
    rows = []
    for w in windows:
        with contextlib.redirect_stdout(io.StringIO()):
            kernel = preprocessor_module.ssp_sir_to_average(call["data"], L, call["fs"], ["automatic", w],
                                                            **call["kwargs"])[3]
        rows.append({"window (ms)": w, "rel. diff to the online kernel":
                     rel_diff(kernel, ONLINE["bundle"]["sspsir_filter_kernel"])})
    return pd.DataFrame(rows).set_index("window (ms)")


REPRO = None
if CONTROL in CALIB:
    REPRO = reproduction_table(CALIB[CONTROL])
    display(REPRO)
    bad = list(REPRO.index[REPRO.reproduced == "NO"])
    if not bad:
        print(f"arm {CONTROL} reproduces the online calibration.")
    else:
        print(f"arm {CONTROL} does NOT reproduce the online calibration: {', '.join(bad)}.")
        if any(o.startswith("dropped by ocular") for o in bad):
            print(f"- the ocular gate dropped other trials: the session may have run with another ocular z threshold "
                  f"(here {CALIB[CONTROL]['pp']._opts['trial_reject_opts']['ocular']['z_threshold']}; 2.0 before "
                  f"f4fd395, 2.5 since). Set SESSION_OVERRIDES['{SUBJECT_ID}']['ocular_z_threshold'].")
        # everything SSP-SIR depends on (the post-stimulus MAD gate and the kept trials come after it)
        upstream = {"dropped by ocular", "dropped by pre-stimulus MAD", "bad channels", "ICA unmixing matrix",
                    "ICA excluded components", "SOUND filter"}
        if "SSP-SIR artifact kernel" in bad and not upstream & set(bad) and CALIB[CONTROL]["sspsir"]:
            probe = sspsir_window_probe(CALIB[CONTROL])
            display(probe.map("{:.1e}".format))
            best = probe["rel. diff to the online kernel"].idxmin()
            used = CALIB[CONTROL]["sspsir"][0]["timerange"]
            if probe.loc[best].iloc[0] < 1e-6:
                print(f"- the online SSP-SIR kernel matches a {best} ms window (this run: {used}). Set "
                      f"SESSION_OVERRIDES['{SUBJECT_ID}'] = {{..., 'sspsir_timerange': ['automatic', {best}]}} "
                      "and run the notebook again.")
            else:
                print("- no window length reproduces the online SSP-SIR kernel; the difference lies elsewhere.")
        print("The arms are still comparable with each other; their numbers describe this decider version.")
else:
    print("The control arm is not among ARMS_TO_RUN: nothing to check.")

### 2.4 Calibration outcomes

Every outcome of `run_calibration()`, one column per arm; section 4.1 adds the online column and marks what
changed. As in `offline_replay.ipynb`:

- **bad channels by detector** are recomputed on the arm's calibration epochs, the post-stimulus detector after the
  arm's low-pass, as in `preprocess_calibration()`. The bundle stores only their union.
- **ICLabel** is recomputed on the ICA input as `calibrate()` builds it. The bundle stores which components were
  excluded, not their probabilities.
- **dipole R² and amplitude** refer to a free-orientation fit at the chosen vertex, on the evoked response of the
  surviving trials, averaged over the chosen samples. Time windows are in ms at the arm's rate.
- **calibrated model**: its predictions on the online calibration model inputs, compared with the online
  `classifier_after_calibration.pt` on the same inputs.

In [ ]:
_CHECKPOINT_MODEL = None


def checkpoint_model(path):
    # One OnlinePredictor whose weights are swapped per checkpoint, loaded as OnlinePredictor.__init__ does.
    # Building it re-seeds numpy and torch; the random state is put back afterwards.
    global _CHECKPOINT_MODEL
    if _CHECKPOINT_MODEL is None:
        state = rng_state()
        _CHECKPOINT_MODEL = OnlinePredictor(GLOBAL_BACKROTATION, model_path=None, seed=SEED)
        set_rng_state(state)
    m = _CHECKPOINT_MODEL
    ck = torch.load(path, map_location=m.device, weights_only=False)
    m.model.wrapped_model.load_state_dict(ck["model_state_dict"])
    align = ck.get("alignment_transform")
    m.model.alignment_transform_torch = align.to(m.device) if align is not None else None
    return m


def bad_channel_detectors(pp):
    # The two detectors of preprocess_calibration(), separately, on the arm's calibration epochs.
    opts, cfg = pp._opts, pp._cfg
    pre, _ = _detect_bad_channels_qc(pp.qc_epochs.copy(), opts["channel_reject_opts"]["pre"], opts["filter_opts"])
    post = pp.post_epochs.copy().apply_baseline(cfg.baseline)
    post = mne.preprocessing.fix_stim_artifact(post, tmin=cfg.artifact_window_1[0], tmax=cfg.artifact_window_1[1],
                                               mode="window")
    if pp.post_lowpass is not None:                   # right after the first blank, as in preprocess_calibration()
        pp.post_lowpass.apply_to_epochs(post)
    post_bad = _detect_bad_channels_post(post, opts["channel_reject_opts"]["post"], cfg.reject_range)
    return [str(c) for c in pre], [str(c) for c in post_bad]


def iclabel(raw_pp, bundle):
    # ICLabel on the ICA input as preprocess_calibration() builds it.
    epochs = raw_pp.ica_epochs.copy()
    if len(bundle["bad_channels"]):
        epochs, _ = _interpolate_bad_channels(epochs, [str(c) for c in bundle["bad_channels"]],
                                              bundle["channel_interpolation_info"])
    f = raw_pp._opts["ica_opts"]["filtering"]
    data, _ = _butter_filter(epochs.get_data(copy=True), f["cutoff"], "bandpass", epochs.info["sfreq"],
                             f["order_bandpass"], f["pad_time_bandpass"])
    epochs = mne.EpochsArray(data, info=epochs.info, events=epochs.events, tmin=epochs.times[0], verbose=False)
    epochs.set_eeg_reference("average", projection=False, verbose=False)
    out = mne_icalabel.label_components(epochs, bundle["ica"], method="iclabel")
    return pd.DataFrame({"label": out["labels"], "probability": np.asarray(out["y_pred_proba"], dtype=float)})


def dipole_summary(src):
    info, fitter = src["fitting_info"], src["fitter"]
    sfreq = float(fitter.sfreq)
    pos, (s, e) = int(info["position_index"]), tuple(int(v) for v in info["index_range"])
    L = fitter._forward["sol"]["data"]
    L = L - L.mean(axis=0)                                     # average-reference leadfield, as fit_trial()
    Lp = L[:, 3 * pos:3 * pos + 3]
    evoked = src["dipole_buffers"].mean(axis=0)
    evoked = evoked - evoked.mean(axis=0)
    r2, amp = [], []
    for i in range(s, e):                                      # free-orientation fit at the chosen vertex
        q = np.linalg.pinv(Lp) @ evoked[:, i]
        r2.append(1 - np.sum((evoked[:, i] - Lp @ q) ** 2) / np.sum(evoked[:, i] ** 2))
        amp.append(np.linalg.norm(q))
    to_ms = lambda i: (DIP_TMIN + i / sfreq) * 1e3
    return dict(vertex=pos, position_mm=fitter._forward["source_rr"][pos] * 1e3,
                orientation=np.asarray(info["orientation"], dtype=float), index_range=(s, e), sfreq=sfreq,
                window_ms=(to_ms(s), to_ms(e - 1)), samples=e - s, r2=float(np.mean(r2)),
                evoked_amplitude=float(np.mean(amp)))


SOURCES = ["online"] + list(ARMS)
DRAW_ORDER = list(ARMS) + ["online"]                 # online drawn last (dashed), so it stays visible on top
CAL_SRC = {"online": ONLINE, **CALIB}
for name, cal in CALIB.items():
    cal["label"] = ARMS[name]["label"]
RAW_PP = next(iter(CALIB.values()))["pp"]           # holds the raw calibration epochs (the ICA input is the same in all arms)

DETECTORS = {name: bad_channel_detectors(cal["pp"]) for name, cal in CALIB.items()}
ICA_LABELS = {src: iclabel(RAW_PP, CAL_SRC[src]["bundle"]) for src in SOURCES}
DIPOLES = {src: dipole_summary(CAL_SRC[src]) for src in SOURCES}

MODELS = {}
ck_path = ONLINE_DIR / "classifier_after_calibration.pt"
if RUN_MODEL and ck_path.is_file():
    m = checkpoint_model(ck_path)
    p_online = m.predict_batch(ONLINE["model_buffers"])
    align_online = m.model.alignment_transform_torch.detach().cpu().numpy()
    MODELS["online"] = dict(mean_p=float(p_online.mean()))
    for name, cal in CALIB.items():
        p = cal["predictor"].predict_batch(ONLINE["model_buffers"])
        align = cal["predictor"].model.alignment_transform_torch.detach().cpu().numpy()
        MODELS[name] = dict(mean_p=float(p.mean()), r=float(stats.pearsonr(p, p_online)[0]),
                            dp=float(np.mean(np.abs(p - p_online))), align=rel_diff(align, align_online))


def outcomes(src):
    s, b = CAL_SRC[src], CAL_SRC[src]["bundle"]
    pp, on = s["pp"], ONLINE["bundle"]
    sf = float(pp.post_sfreq)
    rows = {}

    def add(group, item, value):
        rows[(group, item)] = value

    join = lambda values: ", ".join(str(v) for v in values) if len(values) else "–"
    lp = s["lowpass"]
    add("post window", "rate (Hz)", f"{sf:g}")
    add("post window", "low-pass", lp.describe().replace("post low-pass: ", "") if lp is not None else "none")
    add("post window", "samples in [1, 100] ms", len(np.atleast_1d(b["sspsir_filter_kernel"][0])))

    for gate, label in GATE_NAMES.items():
        add("trials", f"dropped: {label}", len(b["trials_dropped"][gate]))
    add("trials", "used for calibration", len(b["trials_kept"]))
    for gate, label in GATE_NAMES.items():
        add("trials", f"dropped by {label}: trial index", join([int(i) for i in b["trials_dropped"][gate]]))

    add("bad channels", "interpolated", join([str(c) for c in b["bad_channels"]]))
    pre_bad, post_bad = DETECTORS.get(src, ("–", "–"))
    add("bad channels", "flagged pre-stimulus (MAD, power, autocorr.)", join(pre_bad) if src in DETECTORS else "–")
    add("bad channels", "flagged post-stimulus (evoked AUC 20-60 ms)", join(post_bad) if src in DETECTORS else "–")

    ica, labels = b["ica"], ICA_LABELS[src]
    add("ICA", "method / components", f"{ica.method} / {ica.n_components_}")
    add("ICA", "excluded components", join([int(i) for i in ica.exclude]))
    add("ICA", "ICLabel of excluded", join([f"{labels.label[i]} {labels.probability[i]:.3f}" for i in ica.exclude]))
    same_shape = ica.unmixing_matrix_.shape == on["ica"].unmixing_matrix_.shape
    add("ICA", "unmixing matrix: max |Δ| to online",
        "–" if src == "online" else (f"{np.max(np.abs(ica.unmixing_matrix_ - on['ica'].unmixing_matrix_)):.1e}"
                                     if same_shape else "different size"))

    oc = b["ocular_threshold_post"]
    tmin = pp._post_initial_tmin
    add("ocular gate", "z threshold", pp._opts["trial_reject_opts"]["ocular"]["z_threshold"])
    if len(oc):
        idx = np.asarray(next(iter(oc.values()))["time_indices_of_interest"])
        add("ocular gate", "median |z| over (ms)", f"{(tmin + idx[0] / sf) * 1e3:.0f} to {(tmin + idx[-1] / sf) * 1e3:.0f}")
    else:
        add("ocular gate", "median |z| over (ms)", "no ocular components")
    add("ocular gate", "mean / SD taken over (ms)", f"{tmin * 1e3:.0f} to {POST_I_TMAX * 1e3:.0f}")
    for comp, info in oc.items():
        add("ocular gate", f"IC {int(comp)}: mean / SD of |source|", f"{info['mean']:.3f} / {info['std']:.3f}")

    for key, group, gate in (("good_trial_stats_qc", "pre-stimulus MAD gate", "pre"),
                             ("good_trial_stats_post", "post-stimulus MAD gate", "post")):
        st, opts = b[key], pp._opts["trial_reject_opts"][gate]
        add(group, "mean / SD of trial MAD", f"{st['mads_mean']:.3f} / {st['mads_std']:.3f}")
        add(group, "trials in the statistics", len(st["mads"]))
        add(group, "z limits: global / local",
            f"{[float(v) for v in opts['global_zscore_threshold']]} / {opts['local_zscore_threshold']}")

    add("SOUND", "mean diagonal of the filter", f"{np.mean(np.diag(b['sound_filter'])):.4f}")
    add("SOUND", "relative difference to online",
        "–" if src == "online" else f"{rel_diff(b['sound_filter'], on['sound_filter']):.1e}")
    kernel = np.asarray(b["sspsir_filter_kernel"][0])
    add("SSP-SIR", "components removed", n_sspsir(b))
    add("SSP-SIR", "artifact kernel peak (ms)", f"{(POST_TMIN + np.argmax(kernel) / sf) * 1e3:.1f}")
    add("SSP-SIR", "relative difference of P to online",
        "–" if src == "online" else f"{rel_diff(b['sspsir_suppression_matrix_P'], on['sspsir_suppression_matrix_P']):.1e}")

    d = DIPOLES[src]
    add("dipole", "vertex", d["vertex"])
    add("dipole", "position x, y, z (mm)", ", ".join(f"{v:.1f}" for v in d["position_mm"]))
    add("dipole", "orientation x, y, z", ", ".join(f"{v:+.3f}" for v in d["orientation"]))
    add("dipole", "time window (ms)", f"{d['window_ms'][0]:.1f}-{d['window_ms'][1]:.1f}")
    add("dipole", "samples in window", d["samples"])
    add("dipole", "evoked R² at the vertex", f"{d['r2']:.3f}")
    add("dipole", "evoked dipole amplitude (µAm)", f"{d['evoked_amplitude']:.4f}")

    a = s["amplitudes"]
    add("calibration TEP (µAm)", "trials", len(a))
    add("calibration TEP (µAm)", "median / mean / SD", f"{np.median(a):.4f} / {a.mean():.4f} / {a.std():.4f}")
    nz = s["normalizer"]
    add("TEP normaliser", "cal_mean / cal_std (µAm)", f"{nz._cal_mean:.5f} / {nz._cal_std:.5f}")
    add("TEP normaliser", "EWMA span / warm-up trials", f"{nz.ewma_span} / {nz.warmup_period}")
    add("TEP normaliser", "calibration labels: mean / median", f"{np.mean(s['labels']):.3f} / {np.median(s['labels']):.3f}")

    mm = MODELS.get(src, {})
    add("calibrated model", "mean p on the online calibration inputs", f"{mm['mean_p']:.4f}" if mm else "–")
    add("calibrated model", "r with the online model", f"{mm['r']:.4f}" if "r" in mm else "–")
    add("calibrated model", "mean |Δp| to the online model", f"{mm['dp']:.4f}" if "dp" in mm else "–")
    add("calibrated model", "alignment transform: rel. difference to online", f"{mm['align']:.1e}" if "align" in mm else "–")
    add("calibration_summary.csv", "num_calibration_trials", s.get("n_cal_summary", len(b["trials_kept"])))
    return pd.Series(rows)


OUTCOMES = pd.DataFrame({src: outcomes(src) for src in SOURCES}).fillna("–").astype(str)
OUTCOMES.index.names = ["group", "outcome"]
display(OUTCOMES[list(ARMS)])

### 2.5 Dipole position and orientation

The fitted dipole of each arm on the fsaverage source space (grey), in MNE head coordinates. The arrow is the
orientation from `determine_optimal_ori_and_pos()`; section 4.2 overlays all arms and the online dipole.

In [ ]:
SOURCE_RR = ONLINE["fitter"]._forward["source_rr"] * 1e3            # mm
VIEWS = [(0, 1, "top: x right, y anterior"), (0, 2, "front: x right, z superior"), (1, 2, "side: y anterior, z superior")]


def source_label(src):
    return "online log" if src == "online" else ARMS[src]["label"]


def draw_dipoles(axes, names, arrow_mm=25):
    for ax, (h, v, title) in zip(axes, VIEWS):
        ax.scatter(SOURCE_RR[:, h], SOURCE_RR[:, v], s=0.3, color="0.85", rasterized=True)
        for name in names:
            d, st = DIPOLES[name], SOURCE_STYLE[name]
            p, o = d["position_mm"], d["orientation"] * arrow_mm
            ring = name == "online" and len(names) > 1        # online on top: a ring and a thin arrow
            ax.annotate("", xy=(p[h] + o[h], p[v] + o[v]), xytext=(p[h], p[v]),
                        arrowprops=dict(arrowstyle="-|>", color=st["color"], lw=0.9 if ring else 2,
                                        mutation_scale=9 if ring else 14))
            ax.plot(p[h], p[v], st["marker"], color=st["color"], mfc="none" if ring else st["mfc"],
                    ms=16 if ring else 9, mew=1.8,
                    label=f"{source_label(name)}: vertex {d['vertex']}, {d['window_ms'][0]:.1f}-{d['window_ms'][1]:.1f} ms")
        ax.set_xlabel(f"{'xyz'[h]} (mm)")
        ax.set_ylabel(f"{'xyz'[v]} (mm)")
        ax.set_title(title, fontsize=10)
        ax.set_aspect("equal")


for name in ARMS:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    draw_dipoles(axes, [name])
    axes[0].legend(fontsize=8, loc="lower left")
    d = DIPOLES[name]
    fig.suptitle(f"{ARMS[name]['label']}: vertex {d['vertex']}, position {np.round(d['position_mm'], 1)} mm, "
                 f"orientation {np.round(d['orientation'], 3)}, {d['window_ms'][0]:.1f}-{d['window_ms'][1]:.1f} ms",
                 y=1.02, fontsize=10)
    fig.tight_layout()
    plt.show()

### 2.6 Save

Written to each arm's folder: the files `run_calibration()` writes online (`calibration_bundle.npy`, with the frozen
low-pass in arm C, `dipole_fitting_info.npy`, `proc_info.npy` with one rate per window and the low-pass,
`calibration_summary.csv`, `classifier_after_calibration.pt`) and what it does not save: `tep_normalizer.npy`,
`calibration_outcomes.csv`, `calibration_log.txt` and `replay_config.json` (parameters, prime commit, which patches
were applied in memory and the hashes of the files they were applied to, package versions). The folders are
rewritten on every run.

In [ ]:
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "scipy": scipy.__version__,
            "pandas": pd.__version__, "mne": mne.__version__, "mne_icalabel": mne_icalabel.__version__,
            "torch": torch.__version__ if RUN_MODEL else None}
for name, arm in ARMS.items():
    cal, out = CALIB[name], arm["dir"]
    lp = cal["lowpass"]
    np.save(out / "calibration_bundle.npy", cal["bundle"], allow_pickle=True)
    np.save(out / "dipole_fitting_info.npy", cal["fitting_info"], allow_pickle=True)
    np.save(out / "proc_info.npy", {
        "ch_names": list(cal["pp"]._info_processed["ch_names"]),
        "pre": {"tmin": QC_TMIN, "tmax": QC_TMAX, "sfreq": PROC_SFREQ},
        "post": {"tmin": POST_TMIN, "tmax": POST_TMAX, "sfreq": cal["pp"].post_sfreq},
        "dipole": {"tmin": DIP_TMIN, "tmax": DIP_TMAX, "sfreq": cal["pp"].post_sfreq},
        "ica_sfreq": cal["pp"].ica_sfreq,
        "post_lowpass": lp.summary() if lp is not None else None,
        "proc_shape": ("n_channels", "n_times"),
        "units": "as received from the acquisition stream (microvolt-scale, uncalibrated)",
    }, allow_pickle=True)
    pd.DataFrame({"num_calibration_trials": [len(cal["kept"])]}).to_csv(out / "calibration_summary.csv", index=False)
    if RUN_MODEL:
        cal["predictor"].save_checkpoint(out / "classifier_after_calibration.pt")
    nz = cal["normalizer"]
    np.save(out / "tep_normalizer.npy", {
        "cal_mean": nz._cal_mean, "cal_std": nz._cal_std, "ewma_span": nz.ewma_span,
        "warmup_period": nz.warmup_period, "scale_factor": nz.scale_factor, "calibration_trials": cal["kept"],
        "calibration_amplitudes": cal["amplitudes"], "calibration_labels": cal["labels"],
    }, allow_pickle=True)
    OUTCOMES[["online", name]].to_csv(out / "calibration_outcomes.csv")
    (out / "calibration_log.txt").write_text(cal["log"], encoding="utf-8")
    (out / "replay_config.json").write_text(json.dumps({
        "subject": SUBJECT_ID, "session": SESSION_ID, "run": RUN_ID, "arm": name, "arm_label": arm["label"],
        "control_arm": CONTROL, "post_sfreq": arm["post_sfreq"], "ica_sfreq": ICA_SFREQ,
        "post_lowpass": lp.summary() if lp is not None else None, "replay_triplets": REPLAY_TRIPLETS,
        "run_model": RUN_MODEL, "online_dir": str(ONLINE_DIR), "session_overrides": OVERRIDES, "seed": SEED,
        "code": CODE_PROVENANCE, "device": DEVICE_NOTE, "versions": VERSIONS,
        "created": datetime.now().isoformat(timespec="seconds"),
    }, indent=2, default=str))
    print(f"{arm['label']:30s} -> {out}")

## 3 · Intervention replay

### 3.1 Replay

Trial by trial, in session order, as `process_intervention_pulse()` and `analyze_tep()` in `2_intervention.py`, with
each arm's own calibration:

1. **TEP**: `preprocess_post()` on the online `_post_raw` buffer (each arm resamples or low-passes it its own way) →
   `fit_trial()` with the arm's dipole → `TEPNormalizer.transform_verbose()`, which also appends the amplitude to the
   normaliser's EWMA history. Rejected trials get no amplitude and do not enter the history.
2. **Prediction window**: every single pulse (`prime_single_pulse`, forced or not, and `predetermined_single`) → the
   pulse-aligned `_pre_raw` [−210, −10] ms through the arm's own `preprocess_pre()` (interpolation, filter, average
   reference, MAD gates, crop to the model window). For non-forced PRIME trials this is not exactly the window that
   triggered the pulse when the trigger-to-pulse latency was not 10.0 ms (see `offline_replay.ipynb` 4.1), so the
   control arm's predictions deviate slightly from the online log.
3. **Prediction** before the trial's finetuning step, as online. Unlike online, every single pulse whose window passes
   the gates gets one, including forced trials and predetermined trials with a rejected post window.
4. **Finetuning**: `finetune(window, label)` when both exist.

`prime_triplet` trials (with `REPLAY_TRIPLETS`) only go through step 1 without the normaliser: rejection reason and
raw amplitude, as `analyze_tep(update_normalizer=False)` logs them online. They neither update the normaliser nor
finetune, so they do not change anything else. Each post window's processing time (`preprocess_post()` +
`fit_trial()`) is recorded for section 6.3. The model is saved at the end of each block under the online names
(`classifier_after_trial_200.pt`, …).

In [ ]:
ONLINE_COLS = ["prediction_probability", "preprocessing_failed", "postprocessing_failed", "post_reject_reason",
               "tep_amplitude", "tep_amplitude_raw"]
REPLAY_COLS = ["replayed", "post_ms", "postprocessing_failed", "post_reject_reason", "tep_amplitude",
               "tep_amplitude_raw", "tep_ewma", "tep_detrended", "tep_zscore", "pre_source", "preprocessing_failed",
               "pre_reject_reason", "prediction_probability", "finetuned"]


def gate_pre(pp, pre_epoch):
    # The end of Preprocessor.preprocess_pre(): global MAD gate, local MAD gate, crop to the model window.
    stats_qc = pp.calibration_params["good_trial_stats_qc"]
    opts = pp._opts["trial_reject_opts"]["pre"]
    if global_mad_zscore_rejected(pre_epoch, stats_qc["mads_mean"], stats_qc["mads_std"],
                                  opts["global_zscore_threshold"], pp._mad_workspace):
        return None, "global_mad"
    if local_mad_zscore_rejected(pre_epoch, opts["local_zscore_threshold"], pp._mad_workspace):
        return None, "local_mad"
    return model_crop(pre_epoch), None


def pre_from_raw(pp, stem):
    # preprocess_pre_from_pulse(): the pulse-aligned window, with the reason when a gate rejects it.
    pre, epoch = run_preprocess_pre(pp, stem)
    return pre, (None if pre is not None else gate_pre(pp, epoch)[1]), epoch


def replay_intervention(name):
    arm, cal = ARMS[name], CALIB[name]
    pp, fitter, normalizer, predictor = cal["pp"], cal["fitter"], cal["normalizer"], cal["predictor"]
    if RUN_MODEL:
        set_rng_state(cal["rng"])                  # continue the random stream of this arm's calibration
    rows, t0 = [], time.time()
    for block in BLOCKS:
        for r in IV[IV.stage == block].itertuples():
            single = r.condition != "prime_triplet"
            row = dict(stem=r.stem, replayed=bool(single or REPLAY_TRIPLETS))
            if row["replayed"]:
                # analyze_tep(): preprocess_post -> fit_trial (-> transform_verbose, single pulses only)
                tic = time.perf_counter()
                post = pp.preprocess_post(*post_raw(r.stem))
                amplitude = fitter.fit_trial(post) if post is not None else None
                row["post_ms"] = (time.perf_counter() - tic) * 1e3
                row.update(postprocessing_failed=post is None, post_reject_reason=pp.last_post_reject_reason)
                if post is not None and not single:
                    row["tep_amplitude_raw"] = float(amplitude)
                if post is not None and single:
                    tep = normalizer.transform_verbose(amplitude)
                    row.update(tep_amplitude=tep["label"], tep_amplitude_raw=tep["raw"], tep_ewma=tep["ewma"],
                               tep_detrended=tep["detrended"], tep_zscore=tep["zscore"])
                if single and RUN_MODEL:
                    # prediction on the trial's window, before its finetuning step
                    pre, reason, _ = pre_from_raw(pp, r.stem)
                    row.update(pre_source="_pre_raw", preprocessing_failed=pre is None, pre_reject_reason=reason,
                               prediction_probability=predictor.predict(pre) if pre is not None else np.nan)
                    # process_intervention_pulse(): finetune on (window, label) of single pulses
                    row["finetuned"] = bool(pre is not None and post is not None)
                    if row["finetuned"]:
                        predictor.finetune(pre, row["tep_amplitude"])
            rows.append(row)
        if RUN_MODEL:
            predictor.save_checkpoint(arm["dir"] / f"classifier_after_trial_{BLOCK_END[block]}.pt")
    rep = pd.DataFrame(rows).reindex(columns=["stem"] + REPLAY_COLS)
    online = IV[["stem", "stage", "trial_in_stage", "condition", "is_forced", "g", "lat_ms"] + ONLINE_COLS]
    rep = online.rename(columns={c: "online_" + c for c in ONLINE_COLS}).merge(rep, on="stem", how="left")
    n_single = int((rep.replayed.fillna(False).astype(bool) & (rep.condition != "prime_triplet")).sum())
    n_trip = int((rep.replayed.fillna(False).astype(bool) & (rep.condition == "prime_triplet")).sum())
    print(f"{arm['label']}: {n_single} single pulses and {n_trip} triplets in {time.time() - t0:.0f} s"
          + ("; prediction windows from _pre_raw" if RUN_MODEL else ""))
    return rep


REP = {name: replay_intervention(name) for name in ARMS}

### 3.2 Replay against the online log

For the control arm every post-side value should match the online log: rejections exactly, TEP amplitudes and labels
to float precision, for single pulses and triplets alike. Its predictions come from a model that was calibrated and
finetuned again, so GPU arithmetic can make them drift slightly; that drift is the noise floor for comparing arms
with the online log. All arms are summarised per condition.

In [ ]:
def replayed(rep, conditions):
    return rep[rep.replayed.fillna(False).astype(bool) & rep.condition.isin(conditions)]


SINGLE_CONDITIONS = ["predetermined_single", "prime_single_pulse"]


def replay_counts(rep):
    s = replayed(rep, SINGLE_CONDITIONS + ["prime_triplet"])
    g = s.groupby("condition")
    out = pd.DataFrame({"trials": g.size(), "post rejected": g.postprocessing_failed.sum(),
                        "post rejected online": g.online_postprocessing_failed.sum()})
    if RUN_MODEL:
        out["pre rejected"] = g.preprocessing_failed.sum()
        out["predicted"] = g.prediction_probability.count()
        out["predicted online"] = g.online_prediction_probability.count()
        out["finetuned"] = g.finetuned.sum()
        out.loc[out.index == "prime_triplet", ["pre rejected", "predicted", "predicted online", "finetuned"]] = np.nan
    return out


def against_online(rep, conditions, title):
    s = replayed(rep, conditions)
    if s.empty:
        return
    fail, fail_on = s.postprocessing_failed.astype(bool), s.online_postprocessing_failed.astype(bool)
    reason, reason_on = s.post_reject_reason.fillna("kept"), s.online_post_reject_reason.fillna("kept")
    both = s[~fail & ~fail_on]
    raw_rel = (both.tep_amplitude_raw / both.online_tep_amplitude_raw - 1).abs()
    print(title)
    print(f"  postprocessing_failed identical : {int((fail == fail_on).sum())} / {len(s)}")
    print(f"  post_reject_reason identical    : {int((reason == reason_on).sum())} / {len(s)}")
    print(f"  tep_amplitude_raw               : max relative difference {raw_rel.max():.1e} ({len(both)} trials)")
    if "prime_triplet" not in conditions:
        lab = (both.tep_amplitude - both.online_tep_amplitude).abs()
        print(f"  tep_amplitude (label)           : max |diff| {lab.max():.1e}")
        if RUN_MODEL:
            pr = s.dropna(subset=["prediction_probability", "online_prediction_probability"])
            dp = (pr.prediction_probability - pr.online_prediction_probability).abs()
            if len(pr) >= 3:
                first = pr[dp > 0.01].stem.iloc[0] if (dp > 0.01).any() else "none"
                print(f"  prediction_probability          : r = "
                      f"{stats.pearsonr(pr.prediction_probability, pr.online_prediction_probability)[0]:.4f}, "
                      f"median |Δp| {dp.median():.1e}, max |Δp| {dp.max():.1e} ({len(pr)} trials); "
                      f"first |Δp| > 0.01: {first}")


for name, rep in REP.items():
    print(f"── {ARMS[name]['label']}")
    display(replay_counts(rep).astype("Int64"))
if CONTROL in REP:
    against_online(REP[CONTROL], SINGLE_CONDITIONS, f"control arm {CONTROL} vs online log, single pulses")
    against_online(REP[CONTROL], ["prime_triplet"], f"control arm {CONTROL} vs online log, triplets")

### 3.3 Trial tables

One `trials_intervention.csv` per arm, with the rows of the online file (calibration and intervention) and:

- the replayed values under the online column names (`prediction_probability`, `tep_amplitude`,
  `postprocessing_failed`, …), plus `post_ms`, `pre_source`, `pre_reject_reason`, `finetuned` and `replayed`;
- the online values with an `online_` prefix;
- the online timing columns, unchanged, and `trigger_latency_ms`.

Calibration rows carry the calibration amplitude and label of surviving trials, and the gate that dropped a trial in
`calibration_dropped` / `online_calibration_dropped`. Triplet rows carry the replayed rejection and raw amplitude when
`REPLAY_TRIPLETS`.

In [ ]:
TIMING = ["iti", "trial_start_time", "target_time", "max_time", "trigger_time", "pulse_time", "prime_attempts",
          "qc_failures", "threshold", "search_time"]
TIMING = [c for c in TIMING if c in TRIALS]


def trial_table(name):
    cal = CALIB[name]
    gate_of = lambda bundle: {int(i): GATE_NAMES[g] for g, idx in bundle["trials_dropped"].items() for i in idx}
    dropped, dropped_on = gate_of(cal["bundle"]), gate_of(ONLINE["bundle"])
    amp, lab = dict(zip(cal["kept"], cal["amplitudes"])), dict(zip(cal["kept"], cal["labels"]))
    cal_rows = pd.DataFrame([dict(
        stem=r.stem, replayed=True,
        calibration_dropped=dropped.get(r.trial_in_stage, ""),
        online_calibration_dropped=dropped_on.get(r.trial_in_stage, ""),
        preprocessing_failed=dropped.get(r.trial_in_stage) == GATE_NAMES["qc"],
        postprocessing_failed=dropped.get(r.trial_in_stage) in (GATE_NAMES["ocular"], GATE_NAMES["post"]),
        tep_amplitude_raw=amp.get(r.trial_in_stage, np.nan), tep_amplitude=lab.get(r.trial_in_stage, np.nan),
    ) for r in CAL.itertuples()])
    iv_rows = REP[name][["stem"] + REPLAY_COLS]
    table = TRIALS[["stem", "stage", "trial_in_stage", "condition", "is_forced", "trial_order", "lat_ms"] + TIMING]
    table = table.merge(pd.concat([cal_rows, iv_rows], ignore_index=True), on="stem", how="left")
    for c in ONLINE_COLS:
        table["online_" + c] = TRIALS[c].to_numpy()
    table = table.rename(columns={"lat_ms": "trigger_latency_ms"})
    columns = (["stage", "trial_in_stage", "condition", "is_forced", "trial_order"] + REPLAY_COLS
               + ["calibration_dropped"] + ["online_" + c for c in ONLINE_COLS] + ["online_calibration_dropped"]
               + TIMING + ["trigger_latency_ms"])
    return table[columns]


TABLES = {}
for name, arm in ARMS.items():
    TABLES[name] = trial_table(name)
    TABLES[name].to_csv(arm["dir"] / "trials_intervention.csv", index=False)
    print(f"{arm['label']:30s} -> {arm['dir'] / 'trials_intervention.csv'} ({len(TABLES[name])} rows)")
display(TABLES[REF][TABLES[REF].condition == "prime_single_pulse"].head(5))

## 4 · Calibration compared across arms

### 4.1 All outcomes

The table of 2.4 with the online column. The control arm is compared with the online calibration (`≠ online`, every
mark is replay noise); the other arms with the reference arm (`REF`, the control arm when it runs). `≠` marks an
outcome that differs at the precision shown; rows without a value in the column compared with are left unmarked. Rows
that state a difference themselves (relative differences, `|Δ|`) are naturally marked too.

In [ ]:
COMPARISON = OUTCOMES[SOURCES].copy()
for name in ARMS:
    other = "online" if name == CONTROL else REF
    if other == name:
        continue
    differs = (COMPARISON[name] != COMPARISON[other]) & (COMPARISON[other] != "–")
    COMPARISON[f"{name} ≠ {other}"] = np.where(differs, "≠", "")
display(COMPARISON)
COMPARISON.to_csv(RESULTS_ROOT / "calibration_comparison.csv")
for name in ARMS:
    if name != REF:
        col = f"{name} ≠ {REF}"
        print(f"arm {name}: {int((COMPARISON[col] == '≠').sum())} of {len(COMPARISON)} calibration outcomes differ from arm {REF}")

### 4.2 Dipoles

All dipoles in one plot, and how far apart they are: distance between the vertices, angle between the orientations,
and the fitted time windows in ms (the window is chosen from 5× as many samples at 5 kHz).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
draw_dipoles(axes, list(ARMS) + ["online"])
axes[0].legend(fontsize=8, loc="lower left")
fig.suptitle("Dipole position and orientation: online and each arm", y=1.02)
fig.tight_layout()
plt.show()


def angle_deg(a, b):
    return float(np.degrees(np.arccos(np.clip(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)), -1, 1))))


pairs = [(name, "online") for name in ARMS] + [(name, REF) for name in ARMS if name != REF]
if {"B", "C"} <= set(ARMS) and REF not in ("B", "C"):
    pairs.append(("C", "B"))
rows = []
for a, b in pairs:
    da, db = DIPOLES[a], DIPOLES[b]
    (wa0, wa1), (wb0, wb1) = da["window_ms"], db["window_ms"]
    rows.append({"comparison": f"{a} vs {b}", "vertices": f"{da['vertex']} vs {db['vertex']}",
                 "distance (mm)": round(float(np.linalg.norm(da["position_mm"] - db["position_mm"])), 1),
                 "orientation angle (°)": round(angle_deg(da["orientation"], db["orientation"]), 1),
                 "windows (ms)": f"{wa0:.1f}-{wa1:.1f} vs {wb0:.1f}-{wb1:.1f}",
                 "window start shift (ms)": round(wa0 - wb0, 1),
                 "overlap (ms)": round(max(0.0, min(wa1, wb1) - max(wa0, wb0)), 1)})
DIPOLE_DIFF = pd.DataFrame(rows).set_index("comparison")
display(DIPOLE_DIFF)

### 4.3 ICA components

ICLabel label and probability of every component, for the online ICA and each arm's ICA, with the excluded (ocular)
components marked. The ICA input is the same 1 kHz window in every arm, so the ICA only differs if the bad channels
do.

In [ ]:
ica_table = pd.concat({src: ICA_LABELS[src].assign(
    excluded=lambda d, s=src: np.where(np.isin(d.index, CAL_SRC[s]["bundle"]["ica"].exclude), "×", ""))
    for src in SOURCES}, axis=1)
ica_table.index.name = "component"
display(ica_table.round(3))

### 4.4 Gate statistics

The per-trial MAD values that set the pre- and post-stimulus MAD gates (`good_trial_stats_qc` / `_post`, computed on
the calibration trials that survived the earlier gates). The post-stimulus values are taken over [20, 60] ms of the
processed epoch, so they carry each arm's bandwidth.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, key, title in zip(axes, ("good_trial_stats_qc", "good_trial_stats_post"),
                          ("pre-stimulus MAD gate (QC window)", "post-stimulus MAD gate (20-60 ms)")):
    values = {src: np.asarray(CAL_SRC[src]["bundle"][key]["mads"]) for src in DRAW_ORDER}
    bins = np.linspace(min(v.min() for v in values.values()), max(v.max() for v in values.values()), 30)
    for src, v in values.items():
        st = SOURCE_STYLE[src]
        ax.hist(v, bins=bins, histtype="step", lw=2, color=st["color"], ls=st["ls"],
                label=f"{source_label(src)}: mean {v.mean():.2f}, SD {v.std():.2f}, n {len(v)}")
    ax.set_xlabel("trial MAD")
    ax.set_ylabel("trials")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### 4.5 Calibration TEP, SSP-SIR kernel, calibration amplitudes

Left: global field power of the calibration evoked response ([1, 100] ms, surviving trials), with each source's
dipole window shaded. Middle: the SSP-SIR artifact kernel (how strongly each time point is corrected). Right: the
dipole amplitudes of the calibration trials, which the normaliser is fitted to. Each source on its own time grid
(1 ms at 1 kHz, 0.2 ms at 5 kHz).

In [ ]:
def epoch_times_ms(n, sfreq):
    return (POST_TMIN + np.arange(n) / sfreq) * 1e3


fig, axes = plt.subplots(1, 3, figsize=(16, 3.9))
all_amps = np.concatenate([CAL_SRC[x]["amplitudes"] for x in SOURCES])
for src in DRAW_ORDER:
    st, s = SOURCE_STYLE[src], CAL_SRC[src]
    t = epoch_times_ms(s["cal_epochs"].shape[2], s["pp"].post_sfreq)
    gfp = s["cal_epochs"].mean(axis=0).std(axis=0)
    axes[0].plot(t, gfp, color=st["color"], ls=st["ls"], lw=1.6, label=f"{source_label(src)} (n = {len(s['cal_epochs'])})")
    w = DIPOLES[src]["window_ms"]
    axes[0].axvspan(w[0], w[1], color=st["color"], alpha=0.12)
    kernel = np.asarray(s["bundle"]["sspsir_filter_kernel"][0])
    axes[1].plot(epoch_times_ms(len(kernel), s["pp"].post_sfreq), kernel, color=st["color"], ls=st["ls"], lw=1.6,
                 label=source_label(src))
    a = s["amplitudes"]
    axes[2].hist(a, bins=np.linspace(0, np.percentile(all_amps, 99), 30), histtype="step", lw=2, color=st["color"],
                 ls=st["ls"], label=f"{source_label(src)}: median {np.median(a):.4f}")
axes[0].set(xlabel="time (ms)", ylabel="GFP (a.u.)", title="calibration TEP, GFP")
axes[1].set(xlabel="time (ms)", ylabel="kernel", title="SSP-SIR artifact kernel")
axes[2].set(xlabel="dipole amplitude (µAm)", ylabel="trials", title="calibration amplitudes")
for ax in axes:
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### 4.6 What each arm leaves in the post window

Power spectrum of the processed calibration epochs ([1, 100] ms, after SSP-SIR and the second blank, the epochs the
dipole is fitted on), averaged over surviving trials and channels, relative to its mean over 5–80 Hz. Arm A stops at
500 Hz (its Nyquist frequency); arm B keeps everything up to 2.5 kHz; arm C follows B up to about 400 Hz and falls
off above. The dashed line marks the low-pass cutoff.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.9))
for src in DRAW_ORDER:
    s, st = CAL_SRC[src], SOURCE_STYLE[src]
    f, p = signal.periodogram(s["cal_epochs"], fs=s["pp"].post_sfreq, window="hann", axis=-1)
    p = p.mean(axis=(0, 1))
    p = p / p[(f >= 5) & (f <= 80)].mean()
    ax.semilogy(f[1:], p[1:], color=st["color"], ls=st["ls"], lw=1.6, label=source_label(src))
if "C" in ARMS:
    ax.axvline(LOWPASS_HZ, color="0.4", lw=0.9, ls=":")
ax.set(xlabel="frequency (Hz)", ylabel="power relative to 5-80 Hz",
       title="processed calibration epochs [1, 100] ms: mean spectrum")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 5 · Rejections by reason

### 5.1 Calibration

Trials dropped per gate, in the order the gates run (ocular → pre-stimulus MAD → post-stimulus MAD), and how many of
them the online calibration dropped too. Bad channels are listed in 4.1 (`bad channels`).

In [ ]:
rows = []
sets_online = {label: set(int(i) for i in ONLINE["bundle"]["trials_dropped"][gate]) for gate, label in GATE_NAMES.items()}
sets_online["kept"] = set(ONLINE["kept"])
for label, on in sets_online.items():
    row = {"gate": label, "online": len(on)}
    for name in ARMS:
        b = CALIB[name]["bundle"]
        ar = set(CALIB[name]["kept"]) if label == "kept" else set(
            int(i) for i in b["trials_dropped"][next(g for g, l in GATE_NAMES.items() if l == label)])
        row.update({name: len(ar), f"{name}: also online": len(ar & on), f"{name}: only {name}": len(ar - on),
                    f"{name}: only online": len(on - ar)})
    rows.append(row)
CAL_REJECTIONS = pd.DataFrame(rows).set_index("gate")
display(CAL_REJECTIONS)

### 5.2 Single pulses

`post_reject_reason` of the single-pulse trials (`prime_single_pulse` and `predetermined_single`): online against each
arm, in the order the gates run in `preprocess_post()` (ocular → global MAD → local MAD), then by condition and block,
and the trials that switched. Rejections decide which trials enter the normaliser history and the finetuning.

In [ ]:
REASONS = ["kept", "ocular", "global_mad", "local_mad"]


def reason_counts(rep, conditions):
    s = replayed(rep, conditions)
    replay = s.post_reject_reason.fillna("kept").value_counts()
    online = s.online_post_reject_reason.fillna("kept").value_counts()
    return replay, online, len(s)


def reason_table(conditions):
    cols = {}
    for name, rep in REP.items():
        replay, online, n = reason_counts(rep, conditions)
        cols.setdefault("online", online)
        cols[name] = replay
    table = pd.DataFrame(cols).reindex(REASONS).fillna(0).astype(int)
    rejected = table.drop(index="kept").sum()
    percent = 100 * rejected / table.sum()
    table.loc["rejected"] = rejected
    table = table.astype(object)
    table.loc["rejected (%)"] = percent.map("{:.1f}".format)
    return table


REJECTION_TABLE = {"single pulses": reason_table(SINGLE_CONDITIONS)}
for cond in SINGLE_CONDITIONS:
    REJECTION_TABLE[cond] = reason_table([cond])
display(pd.concat(REJECTION_TABLE, names=["trials", "reason"]))

SWITCH_ROWS = []
for name, rep in REP.items():
    s = replayed(rep, SINGLE_CONDITIONS).copy()
    s["online"] = s.online_post_reject_reason.fillna("kept")
    s["replay"] = s.post_reject_reason.fillna("kept")
    print(f"── {ARMS[name]['label']}: post-stimulus rejection of single pulses, online (rows) vs arm (columns)")
    display(pd.crosstab(s.online, s.replay, margins=True))
    rej, rej_on = s.postprocessing_failed.astype(bool), s.online_postprocessing_failed.astype(bool)
    flags = s.assign(rejected_online=rej_on, rejected_replay=rej, both=rej & rej_on,
                     only_online=~rej & rej_on, only_replay=rej & ~rej_on)
    cols = ["rejected_online", "rejected_replay", "both", "only_online", "only_replay"]
    by = [flags.groupby(key)[cols].sum().assign(trials=flags.groupby(key).size())[["trials"] + cols]
          for key in ("condition", "stage")]
    display(pd.concat(by, keys=["condition", "block"]).astype(int))
    only_on, only_rep = s[~rej & rej_on], s[rej & ~rej_on]
    print(f"rejected online only ({len(only_on)}): {', '.join(only_on.stem.head(40))}")
    print(f"rejected in arm {name} only ({len(only_rep)}): {', '.join(only_rep.stem.head(40))}")
    SWITCH_ROWS.append({"arm": name, "single pulses": len(s), "post rejected online": int(rej_on.sum()),
                        "post rejected": int(rej.sum()), "only online": len(only_on), "only this arm": len(only_rep),
                        "pre rejected": int(s.preprocessing_failed.fillna(False).astype(bool).sum()) if RUN_MODEL else np.nan})
REJECTIONS = pd.DataFrame(SWITCH_ROWS).set_index("arm")
display(REJECTIONS)

### 5.3 Triplets

The same for the `prime_triplet` trials (with `REPLAY_TRIPLETS`). With the 10 ms pulse interval set in `19bea17`
the second pulse falls inside the first blank and the third at +20 ms (with the 20 ms interval of earlier sessions:
+20 and +40 ms), inside the [20, 60] ms window the MAD gates read. At 1 kHz the resampler smoothed those pulses; at
5 kHz without a low-pass they stay sharp spikes. Online, triplets neither update the normaliser nor finetune, so their
rejections only change what is logged.

In [ ]:
if REPLAY_TRIPLETS and len(TRIPLETS):
    REJECTION_TABLE["prime_triplet"] = reason_table(["prime_triplet"])
    display(REJECTION_TABLE["prime_triplet"])
    rows = []
    for name, rep in REP.items():
        s = replayed(rep, ["prime_triplet"])
        rows.append(s.groupby("stage").postprocessing_failed.sum().rename(name))
    on = replayed(next(iter(REP.values())), ["prime_triplet"]).groupby("stage").online_postprocessing_failed.sum()
    display(pd.concat([on.rename("online")] + rows, axis=1).astype(int).rename_axis("triplets rejected per block"))
else:
    print("triplets not replayed (REPLAY_TRIPLETS = False)")

### 5.4 Between arms

Single pulses whose rejection changes from the reference arm to another arm: reason in `REF` (rows) against reason in
the other arm (columns). Trials kept in one arm and rejected in the other change the normaliser history and the
finetuning data of that arm.

In [ ]:
for name in ARMS:
    if name == REF:
        continue
    a = replayed(REP[REF], SINGLE_CONDITIONS).set_index("stem").post_reject_reason.fillna("kept")
    b = replayed(REP[name], SINGLE_CONDITIONS).set_index("stem").post_reject_reason.fillna("kept")
    both = pd.concat([a.rename(f"arm {REF}"), b.rename(f"arm {name}")], axis=1)
    print(f"── single pulses: arm {REF} (rows) vs arm {name} (columns)")
    display(pd.crosstab(both[f"arm {REF}"], both[f"arm {name}"], margins=True))
    gained = both.index[(both[f"arm {REF}"] != "kept") & (both[f"arm {name}"] == "kept")]
    lost = both.index[(both[f"arm {REF}"] == "kept") & (both[f"arm {name}"] != "kept")]
    print(f"kept only in arm {name} ({len(gained)}): {', '.join(gained[:30])}")
    print(f"kept only in arm {REF} ({len(lost)}): {', '.join(lost[:30])}")

## 6 · TEP amplitudes across arms

### 6.1 Distributions

The free-orientation dipole amplitude (`tep_amplitude_raw`) of the kept single pulses, and the label the normaliser
makes of it (`tep_amplitude`: EWMA-detrended, z-scored against calibration, mapped through the calibration ECDF).
Raw amplitudes come from each source's own dipole and window, so they are not on a common scale; the labels are,
by construction.

In [ ]:
def kept_singles(src):
    if src == "online":
        d = SINGLES[~SINGLES.postprocessing_failed.astype(bool)]
        return d.set_index("stem")[["condition", "stage", "tep_amplitude_raw", "tep_amplitude"]]
    s = replayed(REP[src], SINGLE_CONDITIONS)
    return s[~s.postprocessing_failed.astype(bool)].set_index("stem")[["condition", "stage", "tep_amplitude_raw",
                                                                       "tep_amplitude"]]


KEPT = {src: kept_singles(src) for src in SOURCES}
rows = []
for src in SOURCES:
    a, lab = KEPT[src].tep_amplitude_raw.astype(float), KEPT[src].tep_amplitude.astype(float)
    rows.append({"source": source_label(src), "kept single pulses": len(a), "raw: median (µAm)": a.median(),
                 "raw: IQR (µAm)": a.quantile(0.75) - a.quantile(0.25), "raw: mean (µAm)": a.mean(),
                 "raw: SD (µAm)": a.std(), "label: mean": lab.mean(), "label: SD": lab.std()})
TEP_SUMMARY = pd.DataFrame(rows).set_index("source")
display(TEP_SUMMARY.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
hi = np.nanpercentile(np.concatenate([KEPT[s].tep_amplitude_raw.astype(float) for s in SOURCES]), 99)
for src in DRAW_ORDER:
    st = SOURCE_STYLE[src]
    axes[0].hist(KEPT[src].tep_amplitude_raw.astype(float), bins=np.linspace(0, hi, 35), histtype="step", lw=2,
                 color=st["color"], ls=st["ls"], label=source_label(src))
    axes[1].hist(KEPT[src].tep_amplitude.astype(float), bins=np.linspace(0, 1, 21), histtype="step", lw=2,
                 color=st["color"], ls=st["ls"], label=source_label(src))
axes[0].set(xlabel="tep_amplitude_raw (µAm)", ylabel="single pulses", title="raw dipole amplitude, kept single pulses")
axes[1].set(xlabel="tep_amplitude (label)", ylabel="single pulses", title="normalised label")
for ax in axes:
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### 6.2 Trial by trial

The same trials in two sources: Pearson r and Spearman ρ of the raw amplitudes and of the labels, over the single
pulses kept in both. `online vs control` is the replay floor; the others show how much each change moves the
single-trial TEP. A trial's raw amplitude can shift by a scale factor between arms (other dipole, other window)
without changing its rank, which is what the label and the finetuning use.

In [ ]:
PAIRS = ([("online", CONTROL)] if CONTROL in ARMS else []) + [(REF, name) for name in ARMS if name != REF]
if {"B", "C"} <= set(ARMS) and REF not in ("B", "C"):
    PAIRS.append(("B", "C"))
rows = []
fig, axes = plt.subplots(2, max(len(PAIRS), 1), figsize=(4.6 * max(len(PAIRS), 1), 8.2), squeeze=False)
for k, (a, b) in enumerate(PAIRS):
    both = KEPT[a].join(KEPT[b], how="inner", lsuffix="_a", rsuffix="_b")
    row = {"pair": f"{a} vs {b}", "kept in both": len(both),
           "kept only in the first": len(KEPT[a].index.difference(KEPT[b].index)),
           "kept only in the second": len(KEPT[b].index.difference(KEPT[a].index))}
    for j, (col, unit) in enumerate((("tep_amplitude_raw", "raw (µAm)"), ("tep_amplitude", "label"))):
        x, y = both[f"{col}_a"].astype(float), both[f"{col}_b"].astype(float)
        ax = axes[j][k]
        for cond, color in CONDITION_COLORS.items():
            m = both.condition_a == cond
            ax.scatter(x[m], y[m], s=12, alpha=0.6, color=color, label=f"{cond} (n={int(m.sum())})")
        if len(both) >= 3:
            r, rho = stats.pearsonr(x, y)[0], stats.spearmanr(x, y)[0]
            row.update({f"{unit}: r": r, f"{unit}: rho": rho})
            ax.set_title(f"{a} vs {b}, {unit}\nr = {r:.3f}, ρ = {rho:.3f}, n = {len(both)}", fontsize=10)
        lo, hi_ = np.nanmin([x.min(), y.min()]), np.nanmax([x.max(), y.max()])
        ax.plot([lo, hi_], [lo, hi_], color="0.5", lw=0.8, ls=":")
        ax.set_xlabel(f"{source_label(a)}")
        ax.set_ylabel(f"{source_label(b)}")
        ax.legend(fontsize=7, loc="upper left")
    rows.append(row)
fig.tight_layout()
plt.show()
TEP_AGREEMENT = pd.DataFrame(rows).set_index("pair")
display(TEP_AGREEMENT.round(4))

### 6.3 Processing time per pulse

Wall-clock time of `preprocess_post()` + `fit_trial()` per post window in this replay, per arm, and the time of
`run_calibration()`. Run on the machine that runs the decider, these are its numbers for the post-stimulus part;
online the decider also runs `preprocess_pre()`, the prediction and finetuning, and writes the buffers.

In [ ]:
rows = []
for name, rep in REP.items():
    for label, conds in (("single pulses", SINGLE_CONDITIONS), ("triplets", ["prime_triplet"])):
        t = replayed(rep, conds).post_ms.astype(float).dropna()
        if len(t):
            rows.append({"arm": ARMS[name]["label"], "trials": label, "n": len(t), "median (ms)": t.median(),
                         "95th percentile (ms)": t.quantile(0.95), "max (ms)": t.max()})
TIMING_TABLE = pd.DataFrame(rows).set_index(["arm", "trials"])
display(TIMING_TABLE.round(1))
print("run_calibration() per arm: " + ", ".join(f"{n} {CALIB[n]['seconds_run_calibration']:.1f} s" for n in ARMS))

fig, ax = plt.subplots(figsize=(7, 3.4))
data = [replayed(REP[n], SINGLE_CONDITIONS + ["prime_triplet"]).post_ms.astype(float).dropna() for n in ARMS]
bp = ax.boxplot(data, showfliers=False, patch_artist=True, medianprops=dict(color="0.15", lw=1.6))
for patch, n in zip(bp["boxes"], ARMS):
    patch.set(facecolor="white", edgecolor=SOURCE_STYLE[n]["color"], lw=1.6)
ax.set_xticks(range(1, len(ARMS) + 1))
ax.set_xticklabels([ARMS[n]["label"] for n in ARMS])
ax.set(ylabel="preprocess_post + fit_trial (ms)", title="processing time per post window")
fig.tight_layout()
plt.show()

## 7 · Prediction performance

Section 7 of `offline_replay.ipynb` for all sources side by side: the online log and each arm. Each source uses its
own predictions, labels and rejections, so n differs between panels; the online log has no prediction for forced
trials, the replays do. The PRIME trials were selected by the online model, so their predictions are
range-restricted for the online source; `predetermined_single` trials are the unselected ones. Skipped with
`RUN_MODEL = False`.

### 7.0 Replayed vs online predictions on the same windows

How far each arm's predictions are from the online model's, trial by trial. For the control arm this is the drift
from recomputing calibration and finetuning; for the other arms it is what their labels and gates changed.

In [ ]:
if RUN_MODEL:
    PERF = {"online": SINGLES.assign(p=SINGLES.prediction_probability, label=SINGLES.tep_amplitude,
                                     raw=SINGLES.tep_amplitude_raw)}
    for name, rep in REP.items():
        s = replayed(rep, SINGLE_CONDITIONS)
        PERF[name] = s.assign(p=s.prediction_probability, label=s.tep_amplitude, raw=s.tep_amplitude_raw)
    PERF = {src: d[["stem", "stage", "condition", "is_forced", "g", "p", "label", "raw"]].astype(
                {"p": float, "label": float, "raw": float}).reset_index(drop=True) for src, d in PERF.items()}

    fig, axes = plt.subplots(1, len(ARMS), figsize=(5.4 * len(ARMS), 4.8), squeeze=False)
    for ax, name in zip(axes[0], ARMS):
        d = replayed(REP[name], SINGLE_CONDITIONS).dropna(subset=["prediction_probability", "online_prediction_probability"])
        for cond, color in CONDITION_COLORS.items():
            dc = d[d.condition == cond]
            ax.scatter(dc.online_prediction_probability, dc.prediction_probability, s=14, alpha=0.6, color=color,
                       label=f"{cond} (n={len(dc)})")
        ax.plot([0, 1], [0, 1], color="0.5", lw=0.8, ls=":")
        if len(d) >= 3:
            r = stats.pearsonr(d.online_prediction_probability.astype(float), d.prediction_probability.astype(float))[0]
            dp = (d.prediction_probability - d.online_prediction_probability).abs()
            ax.set_title(f"{source_label(name)}\nr = {r:.3f}, median |Δp| = {dp.median():.3f}, n = {len(d)}", fontsize=10)
        ax.set_xlabel("online prediction_probability")
        ax.set_ylabel("replayed prediction_probability")
        ax.legend(fontsize=8, loc="upper left")
    fig.tight_layout()
    plt.show()
else:
    print("skipped (RUN_MODEL = False)")

### 7.1a Predictions vs TEP label, `predetermined_single`

In [ ]:
CORR = []


def corr_figure(title, ycol, ylabel, condition=None):
    fig, axes = plt.subplots(1, len(SOURCES), figsize=(5.0 * len(SOURCES), 4.6), squeeze=False)
    for ax, src in zip(axes[0], SOURCES):
        d = PERF[src] if condition is None else PERF[src][PERF[src].condition == condition]
        d = d.dropna(subset=["p", ycol])
        for cond, color in CONDITION_COLORS.items():
            dc = d[d.condition == cond]
            if len(dc):
                ax.scatter(dc.p, dc[ycol], s=14, alpha=0.55, color=color, label=f"{cond} (n={len(dc)})")
        if len(d) >= 3:
            r, p_r = stats.pearsonr(d.p, d[ycol])
            rho, p_rho = stats.spearmanr(d.p, d[ycol])
            b1, b0 = np.polyfit(d.p, d[ycol], 1)
            xs = np.linspace(d.p.min(), d.p.max(), 20)
            ax.plot(xs, b0 + b1 * xs, color="0.25", lw=1.5)
            ax.set_title(f"{source_label(src)}\nr = {r:+.3f} (p = {p_r:.3g})   ρ = {rho:+.3f} (p = {p_rho:.3g})   "
                         f"n = {len(d)}", fontsize=10)
            CORR.append({"analysis": title, "source": src, "n": len(d), "r": r, "p (r)": p_r,
                         "rho": rho, "p (rho)": p_rho})
        else:
            ax.set_title(f"{source_label(src)}\nfewer than 3 trials", fontsize=10)
        ax.set_xlabel("prediction_probability")
        ax.legend(fontsize=8, loc="upper left")
    axes[0][0].set_ylabel(ylabel)
    fig.suptitle(title, y=1.03)
    fig.tight_layout()
    plt.show()


if RUN_MODEL:
    corr_figure("prediction vs TEP label, predetermined_single", "label", "tep_amplitude (label)",
                condition="predetermined_single")
else:
    print("skipped (RUN_MODEL = False)")

### 7.1b Predictions vs raw TEP amplitude, `predetermined_single`

In [ ]:
if RUN_MODEL:
    corr_figure("prediction vs raw TEP amplitude, predetermined_single", "raw", "tep_amplitude_raw (µAm)",
                condition="predetermined_single")
else:
    print("skipped (RUN_MODEL = False)")

### 7.1c Predictions vs TEP label, all single pulses

`prime_single_pulse` (forced included in the replays) and `predetermined_single` together.

In [ ]:
if RUN_MODEL:
    corr_figure("prediction vs TEP label, all single pulses", "label", "tep_amplitude (label)")
else:
    print("skipped (RUN_MODEL = False)")

### 7.1d Per intervention block

The correlation of 7.1c per block, and below for `predetermined_single` only. Each prediction is made before its
trial is used for finetuning, so the per-block values show whether a model's predictions track the label better as
it adapts. About 80 single pulses per block (20 predetermined), so each value is noisy.

In [ ]:
def per_block(condition=None):
    rows = []
    for src in SOURCES:
        for block in BLOCKS:
            d = PERF[src][PERF[src].stage == block]
            if condition is not None:
                d = d[d.condition == condition]
            d = d.dropna(subset=["p", "label"])
            r, rho = ((stats.pearsonr(d.p, d.label)[0], stats.spearmanr(d.p, d.label)[0]) if len(d) >= 4
                      else (np.nan, np.nan))
            rows.append({"source": src, "block": block.replace("intervention_block_", "block "), "n": len(d),
                         "r": r, "rho": rho})
    return pd.DataFrame(rows)


def per_block_figure(table, title):
    display(table.pivot(index="block", columns="source", values=["n", "r", "rho"]).round(3))
    fig, ax = plt.subplots(figsize=(7, 3.6))
    for src in DRAW_ORDER:
        d, st = table[table.source == src], SOURCE_STYLE[src]
        ax.plot(d.block, d.rho, color=st["color"], ls=st["ls"], marker=st["marker"], mfc=st["mfc"], lw=1.8,
                label=source_label(src))
    ax.axhline(0, color="0.4", lw=0.8)
    ax.set_ylabel("Spearman ρ, prediction vs label")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


if RUN_MODEL:
    PER_BLOCK = per_block()
    per_block_figure(PER_BLOCK, "prediction vs TEP label per intervention block, all single pulses")
    PER_BLOCK_PRED = per_block("predetermined_single")
    per_block_figure(PER_BLOCK_PRED, "prediction vs TEP label per intervention block, predetermined_single only")
else:
    print("skipped (RUN_MODEL = False)")

## 8 · Out-of-sample test

The four evaluation blocks (t0, t15, t30, t60; run order 1–4 of `trials_evaluation_<k>.csv`) are open-loop: PRIME
did not select these trials and never trained on them. For each source:

- **TEP**: raw dipole amplitude from the source's own preprocessing and dipole (online: the stored bundle and dipole;
  arms: their replayed calibration);
- **prediction window**: `_pre_raw` through the source's `preprocess_pre()`;
- **models**: the source's checkpoints after calibration and after each intervention block.

Per checkpoint: AUC of the prediction for high vs low TEP (split at the median within each block), pooled over the
four blocks, with a two-sided Mann–Whitney p; Spearman ρ between the prediction and the TEP percentile within its
block; and the mean prediction, which shows score drift. AUC 0.5 means no discrimination. The post-window rejections
of the evaluation trials are counted per source as well. Skipped with `RUN_MODEL = False`.

In [ ]:
CHECKPOINTS = [("after calibration", "classifier_after_calibration.pt")] + [
    (b.replace("intervention_block_", "after block "), f"classifier_after_trial_{BLOCK_END[b]}.pt") for b in BLOCKS]
BLOCK_LABELS = {"evaluation_1": "t0", "evaluation_2": "t15", "evaluation_3": "t30", "evaluation_4": "t60"}
OOS, OOS_TABLE, OOS_BLOCKS = {}, pd.DataFrame(), pd.DataFrame()


def oos_metrics(d, col):
    d = d.dropna(subset=["amplitude", col])
    if len(d) < 6 or d.block.nunique() == 0:
        return {"n": len(d), "AUC": np.nan, "p (AUC)": np.nan, "rho": np.nan, "p (rho)": np.nan,
                "mean p": d[col].mean() if len(d) else np.nan}
    high = d.groupby("block").amplitude.transform(lambda a: a > a.median()).astype(bool)
    pct = d.groupby("block").amplitude.rank(pct=True)
    hi, lo = d.loc[high, col], d.loc[~high, col]
    u = stats.mannwhitneyu(hi, lo, alternative="two-sided")
    rho, p_rho = stats.spearmanr(d[col], pct)
    return {"n": len(d), "AUC": u.statistic / (len(hi) * len(lo)), "p (AUC)": u.pvalue, "rho": rho,
            "p (rho)": p_rho, "mean p": d[col].mean()}


if RUN_MODEL and len(OPEN_TRIALS):
    OOS_SOURCES = {"online": dict(pp=ONLINE["pp"], fitter=ONLINE["fitter"], dir=ONLINE_DIR)}
    OOS_SOURCES.update({name: dict(pp=CALIB[name]["pp"], fitter=CALIB[name]["fitter"], dir=ARMS[name]["dir"])
                        for name in ARMS})
    for src, s in OOS_SOURCES.items():
        t0, rows, windows = time.time(), [], []
        for r in OPEN_TRIALS.itertuples():
            if not all((ONLINE_DIR / f"{r.stem}_{kind}_raw.npy").is_file() for kind in ("pre", "post")):
                continue
            post = s["pp"].preprocess_post(*post_raw(r.stem))
            pre, reason, _ = pre_from_raw(s["pp"], r.stem)
            rows.append({"stem": r.stem, "block": BLOCK_LABELS.get(r.block, r.block),
                         "amplitude": s["fitter"].fit_trial(post) if post is not None else np.nan,
                         "post_reject_reason": s["pp"].last_post_reject_reason, "pre_reject_reason": reason})
            windows.append(pre)
        df = pd.DataFrame(rows)
        ok = np.array([w is not None for w in windows], dtype=bool)
        for label, fname in CHECKPOINTS:
            if ok.any() and (s["dir"] / fname).is_file():
                p = np.full(len(df), np.nan)
                p[ok] = checkpoint_model(s["dir"] / fname).predict_batch(np.stack([w for w in windows if w is not None]))
                df[label] = p
        OOS[src] = df
        print(f"{source_label(src)}: {len(df)} trials, {int(df.amplitude.notna().sum())} with a TEP, {int(ok.sum())} "
              f"with a prediction window; checkpoints: {[c for c, _ in CHECKPOINTS if c in df]} ({time.time() - t0:.0f} s)")

    display(pd.DataFrame({source_label(src): df.post_reject_reason.fillna("kept").value_counts()
                          for src, df in OOS.items()}).fillna(0).astype(int).rename_axis("evaluation trials"))

    rows, block_rows = [], []
    for src, df in OOS.items():
        for label, _ in CHECKPOINTS:
            if label not in df:
                continue
            rows.append({"source": src, "checkpoint": label, **oos_metrics(df, label)})
            for block in df.block.unique():
                block_rows.append({"source": src, "checkpoint": label, "block": block,
                                   **oos_metrics(df[df.block == block], label)})
    OOS_TABLE = pd.DataFrame(rows).set_index(["source", "checkpoint"])
    OOS_BLOCKS = pd.DataFrame(block_rows)
    display(OOS_TABLE.round(3))
    if len(OOS_BLOCKS):
        print("AUC per block:")
        auc = OOS_BLOCKS.pivot_table(index=["source", "checkpoint"], columns="block", values="AUC", dropna=False)
        present = set(zip(OOS_BLOCKS.source, OOS_BLOCKS.checkpoint))
        order = [(src, c) for src in OOS for c, _ in CHECKPOINTS if (src, c) in present]
        display(auc.loc[order, [b for b in BLOCK_LABELS.values() if b in auc.columns]].round(3))

    CKPT_X = {c: i for i, (c, _) in enumerate(CHECKPOINTS)}      # fixed x position per checkpoint
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.9))
    for src in [x for x in DRAW_ORDER if x in OOS_TABLE.index.get_level_values(0)]:
        st, d = SOURCE_STYLE[src], OOS_TABLE.loc[src]
        for ax, col in zip(axes, ("AUC", "rho", "mean p")):
            ax.plot([CKPT_X[c] for c in d.index], d[col], color=st["color"], ls=st["ls"], marker=st["marker"],
                    mfc=st["mfc"], lw=1.8, label=source_label(src))
    axes[0].axhline(0.5, color="0.4", lw=0.8)
    axes[1].axhline(0, color="0.4", lw=0.8)
    axes[0].set(ylabel="AUC, high vs low TEP", title="discrimination (pooled, within-block median split)")
    axes[1].set(ylabel="Spearman ρ", title="prediction vs within-block TEP percentile")
    axes[2].set(ylabel="mean prediction", title="score level")
    for ax in axes:
        ax.set_xticks(list(CKPT_X.values()))
        ax.set_xticklabels(list(CKPT_X), rotation=30)
        ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print("skipped (RUN_MODEL = False)" if not RUN_MODEL else "no evaluation blocks in this session")

### 8.1 Prediction vs raw TEP amplitude

The scatter of 7.1b for the open-loop evaluation trials: prediction against the source's own raw dipole amplitude,
one panel per source, with Pearson r, Spearman ρ and a least-squares line over all trials pooled. Points are coloured
by evaluation block. The plot uses the model of one checkpoint, `OOS_PLOT_CHECKPOINT` (default: after the last
intervention block). The raw amplitudes are not detrended and not on a common scale across sources.

In [ ]:
OOS_SCATTER = pd.DataFrame()
if OOS:
    OOS_PLOT_CHECKPOINT = CHECKPOINTS[-1][0]          # a label from CHECKPOINTS, e.g. "after calibration" or "after block 2"
    BLOCK_COLORS = dict(zip(BLOCK_LABELS.values(), ["#a3bffa", "#5b8ff9", "#4a3aa7", "#2b1f6b"]))   # t0 ... t60, light to dark
    srcs = [s for s in SOURCES if s in OOS and OOS_PLOT_CHECKPOINT in OOS[s]]
    fig, axes = plt.subplots(1, len(srcs), figsize=(5.0 * len(srcs), 4.6), squeeze=False)
    rows = []
    for ax, src in zip(axes[0], srcs):
        d = OOS[src].dropna(subset=[OOS_PLOT_CHECKPOINT, "amplitude"])
        for block in d.block.unique():
            db = d[d.block == block]
            ax.scatter(db[OOS_PLOT_CHECKPOINT], db.amplitude, s=14, alpha=0.55, color=BLOCK_COLORS.get(block, "0.5"),
                       label=f"{block} (n={len(db)})")
        if len(d) >= 3:
            r, p_r = stats.pearsonr(d[OOS_PLOT_CHECKPOINT], d.amplitude)
            rho, p_rho = stats.spearmanr(d[OOS_PLOT_CHECKPOINT], d.amplitude)
            b1, b0 = np.polyfit(d[OOS_PLOT_CHECKPOINT], d.amplitude, 1)
            xs = np.linspace(d[OOS_PLOT_CHECKPOINT].min(), d[OOS_PLOT_CHECKPOINT].max(), 20)
            ax.plot(xs, b0 + b1 * xs, color="0.25", lw=1.5)
            ax.set_title(f"{source_label(src)}\nr = {r:+.3f} (p = {p_r:.3g})   ρ = {rho:+.3f} (p = {p_rho:.3g})   "
                         f"n = {len(d)}", fontsize=10)
            rows.append({"source": src, "checkpoint": OOS_PLOT_CHECKPOINT, "n": len(d), "r": r, "p (r)": p_r,
                         "rho": rho, "p (rho)": p_rho})
        else:
            ax.set_title(f"{source_label(src)}\nfewer than 3 trials", fontsize=10)
        ax.set_xlabel(f"prediction_probability ({OOS_PLOT_CHECKPOINT})")
        ax.legend(fontsize=8, loc="upper left")
    axes[0][0].set_ylabel("tep_amplitude_raw (µAm)")
    fig.suptitle(f"out-of-sample: prediction vs raw TEP amplitude, evaluation blocks, {OOS_PLOT_CHECKPOINT}", y=1.03)
    fig.tight_layout()
    plt.show()
    OOS_SCATTER = pd.DataFrame(rows).set_index(["source", "checkpoint"]) if rows else pd.DataFrame()
    display(OOS_SCATTER.round(4))
else:
    print("skipped")

## Summary

The main numbers of sections 2–8 in one place. The tables are saved in `RESULTS_ROOT`
(`calibration_comparison.csv`, `calibration_rejections.csv`, `rejections_by_reason.csv`, `rejection_switches.csv`,
`dipole_differences.csv`, `tep_summary.csv`, `tep_agreement.csv`, `processing_time.csv`, and with the model
`prediction_correlations*.csv`, `out_of_sample*.csv`); per-arm files are in each arm's folder.

In [ ]:
print("Calibration")
for src in SOURCES:
    b, d = CAL_SRC[src]["bundle"], DIPOLES[src]
    print(f"  {source_label(src):30s} kept {len(b['trials_kept']):3d} "
          f"(ocular {len(b['trials_dropped']['ocular'])}, pre MAD {len(b['trials_dropped']['qc'])}, "
          f"post MAD {len(b['trials_dropped']['post'])}); bad channels {', '.join(map(str, b['bad_channels'])) or '-'}; "
          f"SSP-SIR {n_sspsir(b)} components; dipole vertex {d['vertex']}, {d['window_ms'][0]:.1f}-{d['window_ms'][1]:.1f} ms")
if REPRO is not None:
    print(f"  control arm {CONTROL}: "
          + ("reproduces the online calibration" if (REPRO.reproduced == "yes").all()
             else "does not reproduce: " + ", ".join(REPRO.index[REPRO.reproduced == "NO"])))
display(DIPOLE_DIFF)

print("Rejections by reason")
display(pd.concat(REJECTION_TABLE, names=["trials", "reason"]))
display(REJECTIONS)

print("TEP amplitudes: kept single pulses, and agreement between sources")
display(TEP_SUMMARY.round(4))
display(TEP_AGREEMENT.round(3))

print("Processing time per post window")
display(TIMING_TABLE.round(1))

out = RESULTS_ROOT
CAL_REJECTIONS.to_csv(out / "calibration_rejections.csv")
pd.concat(REJECTION_TABLE, names=["trials", "reason"]).to_csv(out / "rejections_by_reason.csv")
REJECTIONS.to_csv(out / "rejection_switches.csv")
DIPOLE_DIFF.to_csv(out / "dipole_differences.csv")
TEP_SUMMARY.to_csv(out / "tep_summary.csv")
TEP_AGREEMENT.to_csv(out / "tep_agreement.csv")
TIMING_TABLE.to_csv(out / "processing_time.csv")
if REPRO is not None:
    REPRO.to_csv(out / "control_reproduction.csv")
if RUN_MODEL:
    CORR_TABLE = pd.DataFrame(CORR).set_index(["analysis", "source"]) if CORR else pd.DataFrame()
    print("Prediction vs TEP (section 7)")
    display(CORR_TABLE.round(4))
    CORR_TABLE.to_csv(out / "prediction_correlations.csv")
    PER_BLOCK.to_csv(out / "prediction_correlations_by_block.csv", index=False)
    PER_BLOCK_PRED.to_csv(out / "prediction_correlations_by_block_predetermined.csv", index=False)
    if len(OOS_TABLE):
        print("Out-of-sample (section 8)")
        display(OOS_TABLE.round(3))
        OOS_TABLE.to_csv(out / "out_of_sample.csv")
        OOS_BLOCKS.to_csv(out / "out_of_sample_by_block.csv", index=False)
print(f"tables saved to {out}")